# WESAD + SSL TimePatch Transformer

Incluye:
- cargar WESAD
- usar solo BVP, EDA, TEMP wrist
- convertir el problema a stress vs no-stress
- usar un TimePatch Transformer
- hacer self-supervised learning
- generar datos sintéticos parecidos a WESAD para aumentación
- evaluar con LOSO sobre sujetos reales

La idea central es comparar tres escenarios:
- solo supervisado
- SSL + supervisado
- SSL + datos sintéticos + supervisado

In [1]:

!pip -q install numpy scipy scikit-learn matplotlib torch tqdm pandas

In [2]:

import os
import pickle
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


## Configurando la ruta del dataset

OJO: estructura esperada:

```text
WESAD/
  S2/S2.pkl
  S3/S3.pkl
  ...
```

In [3]:

WESAD_ROOT = "/home/leoisidro/CICLOS/IX/TESIS/WESAD-PROCESAMIENTO/WESAD/WESAD"   # OJO actualicen al dataset
RESULTS_DIR = "/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna"

#la frecuencia objetivo:
TARGET_FS = 4
#la longitud de ventana:
WINDOW_SECONDS = 60
#el salto entre ventanas:
WINDOW_STRIDE_SECONDS = 30

USE_LABELS = {1, 2, 3}   # baseline, stress, amusement
STRESS_LABEL = 2
NOSTRESS_LABELS = {1, 3} #convirtiendo a binario

## Cargar WESAD

In [4]:

#Cada señal de WESAD puede venir con distinta frecuencia.
#Esta función la remuestrea a una frecuencia común:
#todo se lleva a 4 Hz
#Esto sirve para que las tres señales tengan la misma longitud temporal y puedan entrar juntas al modelo.
def resample_1d(x, orig_fs, target_fs):
    x = np.asarray(x).squeeze()
    if orig_fs == target_fs:
        return x.astype(np.float32)
    duration = len(x) / float(orig_fs)
    target_len = max(2, int(round(duration * target_fs)))
    return signal.resample(x, target_len).astype(np.float32)

#Esta función normaliza cada señal de forma robusta usando:
#mediana
#rango intercuartílico

#Se usa esto en lugar de una normalización simple con media/desviación porque las señales fisiológicas suelen tener:
#picos
#ruido
#valores atípicos

#Así el modelo aprende patrones más estables entre sujetos.
def robust_norm(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=np.nanmedian(x) if np.isnan(x).any() else 0.0)
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    scale = iqr if iqr > 1e-6 else (np.std(x) + 1e-6)
    return ((x - med) / scale).astype(np.float32)

#Esta función abre el .pkl de cada sujeto y extrae solo:

#BVP
#EDA
#TEMP
#label

#Luego:

#remuestrea cada señal a 4 Hz
#alinea las señales y etiquetas
#normaliza
#deja listo un arreglo temporal con 3 canales
def load_wesad_subject(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    wrist = data["signal"]["wrist"]
    labels = np.asarray(data["label"]).astype(int).squeeze()

    bvp = np.asarray(wrist["BVP"]).squeeze()
    eda = np.asarray(wrist["EDA"]).squeeze()
    temp = np.asarray(wrist["TEMP"]).squeeze()

    fs_bvp = 64
    fs_eda = 4
    fs_temp = 4
    fs_label = 700

    bvp = resample_1d(bvp, fs_bvp, TARGET_FS)
    eda = resample_1d(eda, fs_eda, TARGET_FS)
    temp = resample_1d(temp, fs_temp, TARGET_FS)
    labels = resample_1d(labels, fs_label, TARGET_FS).round().astype(int)

    L = min(len(bvp), len(eda), len(temp), len(labels))
    X = np.stack([
        robust_norm(bvp[:L]),
        robust_norm(eda[:L]),
        robust_norm(temp[:L]),
    ], axis=1)

    labels = labels[:L]
    keep = np.isin(labels, list(USE_LABELS))
    X = X[keep]
    labels = labels[keep]
    y = np.where(labels == STRESS_LABEL, 1, 0).astype(np.int64)
    return X, y

#Después, divide la señal continua en ventanas:

#cada ventana dura 60 s
#con stride de 30 s

#Entonces cada ventana tiene información temporal suficiente para captar respuesta fisiológica al estrés, y además se generan muchas muestras por sujeto.

#Cada ventana queda con forma aproximada:

#tiempo x canales
#o sea: T x 3
def segment_windows(X, y, window_seconds=60, stride_seconds=30, fs=4):
    win = int(window_seconds * fs)
    stride = int(stride_seconds * fs)
    Xw, yw = [], []

    for start in range(0, len(X) - win + 1, stride):
        end = start + win
        xw = X[start:end]
        yw_raw = y[start:end]
        frac = yw_raw.mean()
        if frac in [0.0, 1.0] or frac <= 0.2 or frac >= 0.8:
            Xw.append(xw.astype(np.float32))
            yw.append(int(frac >= 0.5))

    if len(Xw) == 0:
        return np.zeros((0, win, X.shape[1]), dtype=np.float32), np.zeros((0,), dtype=np.int64)

    return np.stack(Xw), np.asarray(yw, dtype=np.int64)

#Se juntan todas las ventanas de todos los sujetos en tres variables:

#X: ventanas
#y: etiquetas binarias
#groups: identificador del sujeto

#groups es clave porque se usa para la evaluación Leave-One-Subject-Out.

def load_all_wesad(root):
    X_all, y_all, groups_all = [], [], []
    subject_dirs = sorted([d for d in os.listdir(root) if d.startswith("S")])

    for subj in subject_dirs:
        pkl_path = os.path.join(root, subj, f"{subj}.pkl")
        if not os.path.exists(pkl_path):
            print("Skipping:", pkl_path)
            continue
        X, y = load_wesad_subject(pkl_path)
        Xw, yw = segment_windows(X, y, WINDOW_SECONDS, WINDOW_STRIDE_SECONDS, TARGET_FS)
        if len(Xw) == 0:
            continue
        sid = int(subj[1:])
        X_all.append(Xw)
        y_all.append(yw)
        groups_all.append(np.full(len(yw), sid))
        print(subj, Xw.shape, "stress ratio=", float(yw.mean()))

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    groups_all = np.concatenate(groups_all, axis=0)
    return X_all, y_all, groups_all

X, y, groups = load_all_wesad(WESAD_ROOT)
print("X:", X.shape, "y:", y.shape, "groups:", groups.shape)
print("Subjects:", np.unique(groups))
print("Stress ratio:", float(y.mean()))

S10 (73, 240, 3) stress ratio= 0.3013698630136986
S11 (71, 240, 3) stress ratio= 0.30985915492957744
S13 (72, 240, 3) stress ratio= 0.2916666666666667
S14 (71, 240, 3) stress ratio= 0.30985915492957744
S15 (72, 240, 3) stress ratio= 0.2916666666666667
S16 (71, 240, 3) stress ratio= 0.30985915492957744
S17 (73, 240, 3) stress ratio= 0.3013698630136986
S2 (67, 240, 3) stress ratio= 0.29850746268656714
S3 (68, 240, 3) stress ratio= 0.29411764705882354
S4 (70, 240, 3) stress ratio= 0.2857142857142857
S5 (70, 240, 3) stress ratio= 0.2714285714285714
S6 (70, 240, 3) stress ratio= 0.3
S7 (71, 240, 3) stress ratio= 0.28169014084507044
S8 (71, 240, 3) stress ratio= 0.29577464788732394
S9 (70, 240, 3) stress ratio= 0.3
X: (1060, 240, 3) y: (1060,) groups: (1060,)
Subjects: [ 2  3  4  5  6  7  8  9 10 11 13 14 15 16 17]
Stress ratio: 0.2962264150943396


## Cargar STRESS-ID

In [5]:
# --- Loader desde processed_data (BVP + EDA + TEMP) para usar con TimePatchTransformer
import pandas as pd
import numpy as np

def robust_norm(x):
    x = np.asarray(x, dtype=np.float32)
    if np.isnan(x).any():
        x = np.nan_to_num(x, nan=np.nanmedian(x) if np.isnan(x).any() else 0.0)
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    scale = iqr if iqr > 1e-6 else (np.std(x) + 1e-6)
    return ((x - med) / scale).astype(np.float32)

def resample_1d(signal, target_len):
    signal = np.asarray(signal, dtype=np.float32)
    if len(signal) == target_len:
        return signal
    if len(signal) < 2:
        return np.full(target_len, float(signal[0]) if len(signal) else 0.0, dtype=np.float32)
    xp = np.linspace(0, 1, len(signal))
    x = np.linspace(0, 1, target_len)
    return np.interp(x, xp, signal).astype(np.float32)

def build_dataset_from_processed(sensor_files, label_col="TASK_LABEL", target_length=int(WINDOW_SECONDS * TARGET_FS)):
    # meta columns (incluye Participant para excluirla de features)
    meta_cols = {"Participant","Activity","PSS_BEFORE","PSS_AFTER","BASELINE_LABEL","TASK_LABEL","axis"}
    frames = {}

    for name, path in sensor_files.items():
        df = pd.read_csv(path, index_col=0)
        # Asegurar columna limpia `Participant` desde el índice (evita duplicados)
        participants_index = df.index.astype(str)
        df = df.reset_index(drop=True)
        df["Participant"] = participants_index

        # Comprobar existencia de label_col
        if label_col not in df.columns:
            raise ValueError(f"label_col '{label_col}' no está en {path}; columnas: {list(df.columns)[:10]}")

        # Columnas de señal: todas las columnas numéricas excepto las meta
        sig_cols = [c for c in df.columns if c not in meta_cols and c not in ("Participant","Activity")]
        
        # --- NORMALIZACIÓN POR SUJETO (PARTICIPANTE) ---
        for part, group in df.groupby("Participant"):
            flat_sig = group[sig_cols].values.astype(np.float32).flatten()
            if np.isnan(flat_sig).any():
                flat_sig = np.nan_to_num(flat_sig, nan=np.nanmedian(flat_sig) if np.isnan(flat_sig).any() else 0.0)
            med = np.median(flat_sig)
            iqr = np.percentile(flat_sig, 75) - np.percentile(flat_sig, 25)
            scale = iqr if iqr > 1e-6 else (np.std(flat_sig) + 1e-6)
            
            norm_vals = (df.loc[group.index, sig_cols].values.astype(np.float32) - med) / scale
            df.loc[group.index, sig_cols] = norm_vals
        # -----------------------------------------------

        frames[name] = df[["Participant","Activity"] + sig_cols + [label_col]].copy()

    # Añadir window index por participant+Activity para emparejar ventanas
    for k, df in frames.items():
        df["window_idx"] = df.groupby(["Participant","Activity"]).cumcount()
        frames[k] = df

    # Construir llave para merge: Participant + Activity + window_idx
    for k in frames:
        frames[k]["_key"] = frames[k]["Participant"].astype(str) + "||" + frames[k]["Activity"].astype(str) + "||" + frames[k]["window_idx"].astype(str)

    # Intersección de llaves comunes entre sensores
    common_keys = set.intersection(*[set(frames[k]["_key"].values) for k in frames])
    common_keys = sorted(common_keys)

    X_list = []
    y_list = []
    participants_list = []

    for key in common_keys:
        channel_data = []
        label_val = None
        part = None
        skip = False

        for name, df in frames.items():
            row = df[df["_key"] == key]
            if row.shape[0] != 1:
                skip = True
                break
            row = row.iloc[0]
            # Seleccionar columnas de señal (las guardadas en frames[name])
            sig_cols = [c for c in df.columns if c not in meta_cols and c not in ("Participant","Activity","window_idx","_key",label_col)]
            sig = row[sig_cols].values.astype(np.float32)
            sig = resample_1d(sig, target_length)
            channel_data.append(sig)

            if label_val is None:
                # Etiquetar basado en Activity en lugar de PSS_AFTER
                activity = row['Activity']
                if activity == 'Baseline':
                    label_val = 0
                else:  # Stroop o Interview
                    label_val = 1
            if part is None:
                part = row["Participant"]

        if skip:
            continue

        X_list.append(np.stack(channel_data, axis=1))
        y_list.append(label_val)
        participants_list.append(part)

    if len(X_list) == 0:
        return np.zeros((0, target_length, len(frames),), dtype=np.float32), np.zeros((0,), dtype=np.int64), np.zeros((0,), dtype=np.int64), []

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    groups = np.array([int(p.lstrip("S")) for p in participants_list], dtype=np.int64)

    return X, y, groups, participants_list

sensor_files = {
    "BVP": f"processed_data/BVP_{WINDOW_SECONDS}_seg.csv",
    "EDA": f"processed_data/EDA_{WINDOW_SECONDS}_seg.csv",
    "TEMP": f"processed_data/TEMP_{WINDOW_SECONDS}_seg.csv",
}

X_proc, y_proc, groups_proc, participants = build_dataset_from_processed(sensor_files, label_col="TASK_LABEL", target_length=int(WINDOW_SECONDS * TARGET_FS))
print("Loaded processed_data -> X.shape:", X_proc.shape, "y.shape:", y_proc.shape, "groups.shape:", groups_proc.shape)

X1, y1, groups_1 = X_proc, y_proc, groups_proc

Loaded processed_data -> X.shape: (1616, 240, 3) y.shape: (1616,) groups.shape: (1616,)


## Datasets y aumentaciones

In [6]:

#Convierte X y y en un formato que PyTorch puede leer con DataLoader.

#Sirve tanto para:

#entrenamiento supervisado
#entrenamiento SSL
#validación
class WindowDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]


#Esta función crea una versión modificada de una ventana.
#Las modificaciones son pequeñas, por ejemplo:

#añadir un poco de ruido
#escalar amplitud
#desplazar un poco en el tiempo
#enmascarar una parte pequeña de un canal

#Esto es fundamental para SSL, porque el modelo ve dos versiones del mismo ejemplo y aprende a reconocer que siguen representando el mismo estado fisiológico.
def weak_augment(x):
    x = x.clone()
    if random.random() < 0.8:
        x = x + 0.02 * torch.randn_like(x)
    if random.random() < 0.5:
        x = x * torch.empty((1, x.shape[1])).uniform_(0.9, 1.1)
    if random.random() < 0.4:
        shift = random.randint(-5, 5)
        x = torch.roll(x, shifts=shift, dims=0)
    if random.random() < 0.3:
        c = random.randrange(x.shape[1])
        start = random.randrange(0, max(1, x.shape[0] - 10))
        width = random.randrange(5, 15)
        x[start:start+width, c] = 0
    return x

#Genera pares (x1, x2) donde:

#x1 = una versión aumentada de la ventana
#x2 = otra versión aumentada de la misma ventana

#Esto se usa para aprendizaje contrastivo.
class SSLPairDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        return weak_augment(x), weak_augment(x)

## TimePatch Transformer

In [7]:

import sys
import os

# Asegurar que TimeMAE esté clonado si estamos en Colab o no existe localmente
if not os.path.exists('TimeMAE'):
    os.system('git clone https://github.com/ustc-time-series/TimeMAE.git')

if 'TimeMAE' not in sys.path:
    sys.path.append('TimeMAE')

import torch
import torch.nn as nn
from TimeMAE.model.TimeMAE import Encoder
from TimeMAE.model.layers import PositionalEmbedding

class TimeMAEPatchEmbed(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128, max_patches=5000):
        super().__init__()
        self.patch_len = patch_len
        self.input_projection = nn.Conv1d(in_ch, emb_dim, kernel_size=patch_len, stride=patch_len)
        self.position = PositionalEmbedding(max_patches, emb_dim)
        
        class Args: pass
        args = Args()
        args.d_model = emb_dim
        args.attn_heads = 4
        args.layers = 2
        args.dropout = 0.1
        args.enable_res_parameter = 1
        self.encoder = Encoder(args)

    def forward(self, x):
        B, T, C = x.shape
        P = self.patch_len
        T2 = (T // P) * P
        x = x[:, :T2, :]
        x = self.input_projection(x.transpose(1, 2)).transpose(1, 2).contiguous()
        pos_emb = self.position(x)
        x += pos_emb[:, :x.size(1), :]
        x = self.encoder(x)
        return x

#En vez de meter toda la secuencia de golpe, la divide en pequeños bloques temporales llamados patches.

#Por ejemplo:

#si patch_len = 8
#entonces cada 8 pasos temporales se agrupan en un patch

#Cada patch se aplana y se proyecta a un vector latente.

#¿Por qué se hace esto?

#reduce longitud efectiva de la secuencia
#facilita que el Transformer trabaje mejor
#captura patrones locales antes de modelar dependencias largas
class PatchEmbed1D(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128):
        super().__init__()
        self.patch_len = patch_len
        self.proj = nn.Linear(in_ch * patch_len, emb_dim)

    def forward(self, x):
        B, T, C = x.shape
        P = self.patch_len
        T2 = (T // P) * P
        x = x[:, :T2, :].reshape(B, T2 // P, P * C)
        return self.proj(x)

#Después del patching, el modelo añade:

#un CLS token
#embeddings posicionales
#un TransformerEncoder

#Luego, el vector CLS se usa como resumen global de la ventana.

#El modelo tiene dos salidas implícitas:

#embedding para SSL
#clasificación para stress/no-stress

#En otras palabras, el Transformer aprende una representación temporal multimodal de la ventana.
class TimePatchTransformer(nn.Module):
    def __init__(self, in_ch=3, patch_len=8, emb_dim=128, depth=4, heads=4, dropout=0.1, num_classes=2):
        super().__init__()
        self.patch = TimeMAEPatchEmbed(in_ch, patch_len, emb_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))
        self.pos_emb = nn.Parameter(torch.zeros(1, 512, emb_dim))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(emb_dim)
        self.ssl_head = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, 64)
        )
        self.cls_head = nn.Linear(emb_dim, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward_features(self, x):
        x = self.patch(x)
        B, N, D = x.shape
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_emb[:, :N+1, :]
        x = self.encoder(x)
        x = self.norm(x)
        return x[:, 0]

    def forward_ssl(self, x):
        z = self.forward_features(x)
        z = self.ssl_head(z)
        return F.normalize(z, dim=-1)

    def forward(self, x):
        z = self.forward_features(x)
        return self.cls_head(z)

## SSL y aprendizaje supervisado

In [8]:
#Esta es la pérdida contrastiva.

#La idea es:

#dos vistas aumentadas del mismo ejemplo deben quedar cerca en el espacio latente
#vistas de ejemplos distintos deben quedar lejos

#Eso obliga al modelo a aprender representaciones útiles sin usar etiquetas.
def nt_xent(z1, z2, temp=0.2):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=-1) / temp
    sim.fill_diagonal_(-1e9)
    targets = torch.arange(B, device=z.device)
    targets = torch.cat([targets + B, targets], dim=0)
    return F.cross_entropy(sim, targets)

#Aquí se hace el preentrenamiento self-supervised.
#Pasos:

#se toman ventanas sin usar la etiqueta
#se generan dos vistas aumentadas
#el modelo produce embeddings para ambas
#se calcula la pérdida contrastiva
#se actualizan los pesos

#Resultado: el modelo aprende una representación inicial buena de las señales fisiológicas

def pretrain_ssl(model, X_unlabeled, epochs=6, batch_size=64, lr=1e-3):
    ds = SSLPairDataset(X_unlabeled)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    model.train()
    for ep in range(epochs):
        losses = []
        for x1, x2 in dl:
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            z1 = model.forward_ssl(x1)
            z2 = model.forward_ssl(x2)
            loss = nt_xent(z1, z2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        print(f"SSL epoch {ep+1}/{epochs}: {np.mean(losses):.4f}")
    return model

def evaluate_probs(y_true, probs):
    pred = (probs >= 0.5).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
    }
    try:
        out["auroc"] = roc_auc_score(y_true, probs)
    except Exception:
        out["auroc"] = np.nan
    return out

def finetune(model, X_train, y_train, X_val, y_val, epochs=8, batch_size=64, lr=2e-4):
    train_dl = DataLoader(WindowDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(WindowDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_f1 = -1

    for ep in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        probs_all, ys_all = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(DEVICE)
                probs = torch.softmax(model(xb), dim=-1)[:, 1].cpu().numpy()
                probs_all.append(probs)
                ys_all.append(yb.numpy())

        probs_all = np.concatenate(probs_all)
        ys_all = np.concatenate(ys_all)
        metrics = evaluate_probs(ys_all, probs_all)
        print(f"FT epoch {ep+1}/{epochs}: loss={np.mean(train_losses):.4f}, val_f1={metrics['f1']:.4f}")

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

def predict_probs(model, X_test, batch_size=128):
    dl = DataLoader(WindowDataset(X_test), batch_size=batch_size, shuffle=False)
    model.eval()
    probs = []
    with torch.no_grad():
        for xb in dl:
            xb = xb.to(DEVICE)
            p = torch.softmax(model(xb), dim=-1)[:, 1].cpu().numpy()
            probs.append(p)
    return np.concatenate(probs)

## Experimentos LOSO sobre sujetos reales de WESAD

In [9]:
#Leave-One-Group-Out, donde el grupo es el sujeto.

#Eso significa:

#en cada fold, se deja un sujeto entero para test
#el entrenamiento se hace con los demás sujetos

#Esto evita “hacer trampa” mezclando ventanas del mismo sujeto en train y test.

#Dentro de cada fold se hace esto:
#a) Separar train y test
#X_train, y_train: todos los sujetos menos uno
#X_test, y_test: el sujeto dejado fuera

#b) Separar validación dentro del train
#Una pequeña parte del entrenamiento se usa como validación.

#c) Opción: añadir sintéticos
#Si use_synth=True:
#se generan ventanas sintéticas
#se añaden al conjunto de entrenamiento real

#d) Opción: hacer SSL
#Si use_ssl=True:
#primero se preentrena con las ventanas de entrenamiento
#luego se hace fine-tuning supervisado

#e) Evaluación
#Al final se calcula el rendimiento en el sujeto real no visto.
#Se reportan métricas como:
#accuracy
#F1
#balanced accuracy
#AUROC

def run_loso(X, y, groups, use_ssl=False, use_synth=False, ssl_epochs=6, ft_epochs=8, synth_ratio=0.5, emb_dim=128,depth=4,heads=4,dropout=0.1):
    logo = LeaveOneGroupOut()
    results = []

    for fold, (tr_idx, te_idx) in enumerate(logo.split(X, y, groups), start=1):
        held_out = int(np.unique(groups[te_idx])[0])
        X_train, y_train = X[tr_idx], y[tr_idx]
        X_test, y_test = X[te_idx], y[te_idx]

        n_val = max(8, int(0.15 * len(X_train)))
        perm = np.random.permutation(len(X_train))
        val_idx = perm[:n_val]
        tr2_idx = perm[n_val:]

        X_tr, y_tr = X_train[tr2_idx], y_train[tr2_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]

        if use_synth:
            Xs, ys = generate_synthetic_dataset(int(len(X_tr) * synth_ratio), class_balance=float(y_tr.mean()))
            X_tr = np.concatenate([X_tr, Xs], axis=0)
            y_tr = np.concatenate([y_tr, ys], axis=0)

        model = TimePatchTransformer(in_ch=3, patch_len=8, emb_dim=emb_dim, depth=depth, heads=heads,dropout=dropout, num_classes=2)

        if use_ssl:
            X_unlab = X_train.copy()
            if use_synth:
                Xu, _ = generate_synthetic_dataset(int(0.5 * len(X_train)), class_balance=float(y_train.mean()))
                X_unlab = np.concatenate([X_unlab, Xu], axis=0)
            model = pretrain_ssl(model, X_unlab, epochs=ssl_epochs)

        model = finetune(model, X_tr, y_tr, X_val, y_val, epochs=ft_epochs)
        probs = predict_probs(model, X_test)
        metrics = evaluate_probs(y_test, probs)
        metrics["subject"] = held_out
        results.append(metrics)
        print("Fold", fold, "subject", held_out, metrics)

    return results

from scipy.stats import t as _t_dist

def summarize(name, results, txt_path="", append=True, alpha=0.05):
    """Resumen de resultados con media, std e intervalo de confianza al 95%.

    Parameters
    ----------
    name     : str   — nombre del experimento.
    results  : list of dict — cada dict tiene claves 'f1', 'balanced_accuracy', 'auroc', 'accuracy'.
    txt_path : str   — ruta de archivo de texto donde guardar los resultados (opcional).
    append   : bool  — si True, agrega al archivo; si False, lo sobreescribe.
    alpha    : float — nivel de significancia (default 0.05 → IC 95%).

    Returns
    -------
    dict con { métrica: { mean, std, ci95_low, ci95_high, n } }
    """
    lines = ["", name]
    print("\n" + "="*70)
    print(name)
    print("="*70)
    summary = {}

    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        vals = np.array([r[key] for r in results], dtype=float)
        vals = vals[~np.isnan(vals)]
        n        = len(vals)
        mean_val = float(np.mean(vals))
        std_val  = float(np.std(vals, ddof=1))          # std muestral
        sem      = std_val / np.sqrt(n)
        t_crit   = float(_t_dist.ppf(1 - alpha / 2, df=max(n - 1, 1)))
        ci_half  = t_crit * sem
        summary[key] = {
            "mean":      mean_val,
            "std":       std_val,
            "ci95_low":  float(mean_val - ci_half),
            "ci95_high": float(mean_val + ci_half),
            "n":         n,
        }
        line = (f"  {key:22s}: {mean_val:.4f} ± {std_val:.4f}"
                f"  [IC95%: ({mean_val - ci_half:.4f}, {mean_val + ci_half:.4f})]  n={n}")
        lines.append(line)
        print(line)

    mode = "a" if append else "w"
    if txt_path:
        with open(txt_path, mode, encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")

    return summary

### Comparación de dos escenarios

Luego el notebook ejecuta dos experimentos completos:

1) Supervised only: Entrena directamente con etiquetas, sin SSL ni sintéticos Esto es baseline principal.

2) SSL + supervised: Primero hace self-supervised pretraining y luego fine-tuning supervisado. Esto sirve para comprobar si SSL ayuda.


### Pruebas Iniciales WESAD

In [75]:

# 1) Supervised only
res_sup = run_loso(X, y, groups, use_ssl=False, use_synth=False, ft_epochs=8)
summarize("Supervised only", res_sup)

FT epoch 1/8: loss=0.3105, val_f1=0.8182
FT epoch 2/8: loss=0.2313, val_f1=0.8381
FT epoch 3/8: loss=0.2259, val_f1=0.8302
FT epoch 4/8: loss=0.1910, val_f1=0.8302
FT epoch 5/8: loss=0.1784, val_f1=0.8393
FT epoch 6/8: loss=0.1846, val_f1=0.8571
FT epoch 7/8: loss=0.1403, val_f1=0.8468
FT epoch 8/8: loss=0.1100, val_f1=0.9159
Fold 1 subject 2 {'accuracy': 0.9850746268656716, 'f1': 0.975609756097561, 'balanced_accuracy': 0.9893617021276595, 'auroc': 1.0, 'subject': 2}
FT epoch 1/8: loss=0.3083, val_f1=0.8095
FT epoch 2/8: loss=0.2215, val_f1=0.8140
FT epoch 3/8: loss=0.1932, val_f1=0.8132
FT epoch 4/8: loss=0.1842, val_f1=0.8276
FT epoch 5/8: loss=0.1676, val_f1=0.8478
FT epoch 6/8: loss=0.1802, val_f1=0.8764
FT epoch 7/8: loss=0.1333, val_f1=0.8989
FT epoch 8/8: loss=0.1213, val_f1=0.8989
Fold 2 subject 3 {'accuracy': 0.75, 'f1': 0.6666666666666666, 'balanced_accuracy': 0.7791666666666667, 'auroc': 0.8291666666666667, 'subject': 3}
FT epoch 1/8: loss=0.3383, val_f1=0.8200
FT epoch 2/8:

KeyboardInterrupt: 

In [21]:

# 2) SSL + supervised
res_ssl = run_loso(X, y, groups, use_ssl=True, use_synth=False, ssl_epochs=6, ft_epochs=8)
summarize("SSL + supervised", res_ssl)

SSL epoch 1/6: 2.5932
SSL epoch 2/6: 2.1152
SSL epoch 3/6: 1.9551
SSL epoch 4/6: 1.7837
SSL epoch 5/6: 1.7520
SSL epoch 6/6: 1.6619
FT epoch 1/8: loss=0.3247, val_f1=0.8544
FT epoch 2/8: loss=0.2384, val_f1=0.8302
FT epoch 3/8: loss=0.1879, val_f1=0.8000
FT epoch 4/8: loss=0.1776, val_f1=0.8485
FT epoch 5/8: loss=0.1508, val_f1=0.8454
FT epoch 6/8: loss=0.1232, val_f1=0.8387
FT epoch 7/8: loss=0.1159, val_f1=0.8980
FT epoch 8/8: loss=0.0964, val_f1=0.8713
Fold 1 subject 2 {'accuracy': 1.0, 'f1': 1.0, 'balanced_accuracy': 1.0, 'auroc': 1.0, 'subject': 2}
SSL epoch 1/6: 2.6747
SSL epoch 2/6: 2.1688
SSL epoch 3/6: 1.9760
SSL epoch 4/6: 1.8331
SSL epoch 5/6: 1.7474
SSL epoch 6/6: 1.5927
FT epoch 1/8: loss=0.3453, val_f1=0.9451
FT epoch 2/8: loss=0.2040, val_f1=0.9263
FT epoch 3/8: loss=0.1578, val_f1=0.9362
FT epoch 4/8: loss=0.1250, val_f1=0.9318
FT epoch 5/8: loss=0.0924, val_f1=0.9263
FT epoch 6/8: loss=0.0629, val_f1=0.9474
FT epoch 7/8: loss=0.0507, val_f1=0.9792
FT epoch 8/8: loss=0.

### Pruebas Iniciales STRESS-ID

In [9]:
# 1) Supervised only
res_sup_1 = run_loso(X1, y1, groups_1, use_ssl=False, use_synth=False, ft_epochs=8)
summarize("Supervised only", res_sup_1)

FT epoch 1/8: loss=0.4282, val_f1=0.8417
FT epoch 2/8: loss=0.3833, val_f1=0.8476
FT epoch 3/8: loss=0.3774, val_f1=0.8476
FT epoch 4/8: loss=0.3733, val_f1=0.8414
FT epoch 5/8: loss=0.3691, val_f1=0.8281
FT epoch 6/8: loss=0.3743, val_f1=0.8325
FT epoch 7/8: loss=0.3633, val_f1=0.8359
FT epoch 8/8: loss=0.3585, val_f1=0.8354
Fold 1 subject 2 {'accuracy': 0.9523809523809523, 'f1': 0.9606299212598425, 'balanced_accuracy': 0.9621212121212122, 'auroc': 0.998057498057498, 'subject': 2}
FT epoch 1/8: loss=0.4274, val_f1=0.8576
FT epoch 2/8: loss=0.3811, val_f1=0.8571
FT epoch 3/8: loss=0.3842, val_f1=0.8547
FT epoch 4/8: loss=0.3718, val_f1=0.8586
FT epoch 5/8: loss=0.3645, val_f1=0.8571
FT epoch 6/8: loss=0.3609, val_f1=0.8542
FT epoch 7/8: loss=0.3525, val_f1=0.8585
FT epoch 8/8: loss=0.3656, val_f1=0.8523
Fold 2 subject 3 {'accuracy': 0.896551724137931, 'f1': 0.9203539823008849, 'balanced_accuracy': 0.8714285714285714, 'auroc': 0.9368131868131868, 'subject': 3}
FT epoch 1/8: loss=0.4326,

In [ ]:
res_ssl_1 = run_loso(X1, y1, groups_1, use_ssl=True, use_synth=False, ssl_epochs=6, ft_epochs=8)
summarize("SSL + supervised", res_ssl_1)

SSL epoch 1/6: 2.0655
SSL epoch 2/6: 1.5722
SSL epoch 3/6: 1.4120
SSL epoch 4/6: 1.2723
SSL epoch 5/6: 1.2094
SSL epoch 6/6: 1.1578
FT epoch 1/8: loss=0.6969, val_f1=0.6971
FT epoch 2/8: loss=0.6628, val_f1=0.6451
FT epoch 3/8: loss=0.6558, val_f1=0.6817
FT epoch 4/8: loss=0.6540, val_f1=0.7078
FT epoch 5/8: loss=0.6500, val_f1=0.6007
FT epoch 6/8: loss=0.6492, val_f1=0.6525
FT epoch 7/8: loss=0.6386, val_f1=0.7124
FT epoch 8/8: loss=0.6352, val_f1=0.7268
Fold 1 subject 2 {'accuracy': 0.638095238095238, 'f1': 0.7285714285714285, 'balanced_accuracy': 0.5914918414918415, 'auroc': 0.6142191142191142, 'subject': 2}
SSL epoch 1/6: 2.0711
SSL epoch 2/6: 1.5543
SSL epoch 3/6: 1.3659
SSL epoch 4/6: 1.3029
SSL epoch 5/6: 1.2172
SSL epoch 6/6: 1.1484
FT epoch 1/8: loss=0.6837, val_f1=0.7084
FT epoch 2/8: loss=0.6630, val_f1=0.7260
FT epoch 3/8: loss=0.6585, val_f1=0.7136
FT epoch 4/8: loss=0.6522, val_f1=0.6957
FT epoch 5/8: loss=0.6444, val_f1=0.6990
FT epoch 6/8: loss=0.6367, val_f1=0.7096
FT 

KeyboardInterrupt: 

## Optimizacion de hiperparámetros con Optuna

In [76]:
import os
import json
from datetime import datetime

import numpy as np
import torch
import optuna


def objective_optuna(trial, X, y, groups):
    params = {
        'patch_len': trial.suggest_categorical('patch_len', [8, 16]),
        'emb_dim': trial.suggest_categorical('emb_dim', [128, 256]),
        'depth': trial.suggest_int('depth', 3, 6),
        'dropout': trial.suggest_float('dropout', 0.02, 0.25),
        'ft_epochs': trial.suggest_int('ft_epochs', 6, 10),
        'ssl_epochs': trial.suggest_int('ssl_epochs', 4, 10),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32]),
        'use_ssl': trial.suggest_categorical('use_ssl', [True]),
    }

    torch.cuda.empty_cache()
    results = run_loso(
        X, y, groups,
        use_ssl=params['use_ssl'],
        ssl_epochs=params['ssl_epochs'],
        ft_epochs=params['ft_epochs'],
        synth_ratio=0.5,
        emb_dim=params['emb_dim'],
        depth=params['depth'],
        dropout=params['dropout'],
    )

    mean_f1 = float(np.nanmean([r['f1'] for r in results]))
    trial.set_user_attr('mean_accuracy', float(np.nanmean([r['accuracy'] for r in results])))
    trial.set_user_attr('mean_auroc', float(np.nanmean([r.get('auroc', np.nan) for r in results])))
    trial.set_user_attr('params', params)
    return mean_f1


def run_optuna_search(X, y, groups, n_trials=20, study_name='wesad_optuna'):
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction='maximize', sampler=sampler, study_name=study_name)

    def _objective(trial):
        return objective_optuna(trial, X, y, groups)

    study.optimize(_objective, n_trials=n_trials)

    print('\n' + '=' * 80)
    print('OPTUNA FINALIZADO')
    print('=' * 80)
    print(f'Best F1: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')

    best_path = os.path.join(RESULTS_DIR, f'{study_name}.json')
    payload = {
        'name': study_name,
        'best_value': float(study.best_value),
        'best_params': study.best_params,
        'trials': [
            {
                'number': t.number,
                'value': None if t.value is None else float(t.value),
                'params': t.params,
                'user_attrs': t.user_attrs,
                'state': str(t.state),
            }
            for t in study.trials
        ],
    }
    with open(best_path, 'w') as f:
        json.dump(payload, f, indent=2)

    print(f'Resultados guardados en: {best_path}')
    return study


print('Optuna listo')

Optuna listo


### Mejores Hiperparámetros WESAD

In [77]:
# EJECUTAR OPTUNA
print("INICIANDO OPTUNA...")
study_optuna = run_optuna_search(
    X, y, groups,
    n_trials=5,
    study_name=f"wesad_optuna_{WINDOW_SECONDS}seg"
)

print("\nMejor F1:", study_optuna.best_value)
print("Mejores parámetros:", study_optuna.best_params)

[I 2026-05-26 21:32:52,838] A new study created in memory with name: wesad_optuna_60seg


INICIANDO OPTUNA...
SSL epoch 1/10: 2.6026
SSL epoch 2/10: 2.2586
SSL epoch 3/10: 2.0282
SSL epoch 4/10: 1.8871
SSL epoch 5/10: 1.7751
SSL epoch 6/10: 1.7045
SSL epoch 7/10: 1.6144
SSL epoch 8/10: 1.5506
SSL epoch 9/10: 1.5116
SSL epoch 10/10: 1.4613
FT epoch 1/6: loss=0.3661, val_f1=0.8909
FT epoch 2/6: loss=0.2286, val_f1=0.8909
FT epoch 3/6: loss=0.1888, val_f1=0.9107
FT epoch 4/6: loss=0.1586, val_f1=0.9174
FT epoch 5/6: loss=0.1203, val_f1=0.9204
FT epoch 6/6: loss=0.0990, val_f1=0.9515
Fold 1 subject 2 {'accuracy': 1.0, 'f1': 1.0, 'balanced_accuracy': 1.0, 'auroc': 1.0, 'subject': 2}
SSL epoch 1/10: 2.6380
SSL epoch 2/10: 2.2417
SSL epoch 3/10: 2.0845
SSL epoch 4/10: 1.9268
SSL epoch 5/10: 1.8641
SSL epoch 6/10: 1.7520
SSL epoch 7/10: 1.6669
SSL epoch 8/10: 1.6164
SSL epoch 9/10: 1.5317
SSL epoch 10/10: 1.5137
FT epoch 1/6: loss=0.2753, val_f1=0.7750
FT epoch 2/6: loss=0.1787, val_f1=0.7848
FT epoch 3/6: loss=0.1568, val_f1=0.8148
FT epoch 4/6: loss=0.1422, val_f1=0.8250
FT epoch

[I 2026-05-26 21:34:03,559] Trial 0 finished with value: 0.7595277674408643 and parameters: {'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.05587873967732661, 'ft_epochs': 6, 'ssl_epochs': 10, 'batch_size': 32, 'use_ssl': True}. Best is trial 0 with value: 0.7595277674408643.


FT epoch 6/6: loss=0.0874, val_f1=0.9333
Fold 15 subject 17 {'accuracy': 0.4657534246575342, 'f1': 0.0, 'balanced_accuracy': 0.3333333333333333, 'auroc': 0.12745098039215685, 'subject': 17}
SSL epoch 1/7: 2.6582
SSL epoch 2/7: 2.2300
SSL epoch 3/7: 1.9995
SSL epoch 4/7: 1.8580
SSL epoch 5/7: 1.7430
SSL epoch 6/7: 1.6659
SSL epoch 7/7: 1.5559
FT epoch 1/7: loss=0.4652, val_f1=0.8411
FT epoch 2/7: loss=0.2491, val_f1=0.8824
FT epoch 3/7: loss=0.2236, val_f1=0.8800
FT epoch 4/7: loss=0.1880, val_f1=0.9000
FT epoch 5/7: loss=0.1515, val_f1=0.8911
FT epoch 6/7: loss=0.1193, val_f1=0.9200
FT epoch 7/7: loss=0.0898, val_f1=0.9400
Fold 1 subject 2 {'accuracy': 0.9850746268656716, 'f1': 0.975609756097561, 'balanced_accuracy': 0.9893617021276595, 'auroc': 1.0, 'subject': 2}
SSL epoch 1/7: 2.5556
SSL epoch 2/7: 2.0458
SSL epoch 3/7: 1.8530
SSL epoch 4/7: 1.7292
SSL epoch 5/7: 1.6771
SSL epoch 6/7: 1.6068
SSL epoch 7/7: 1.5091
FT epoch 1/7: loss=0.3410, val_f1=0.8817
FT epoch 2/7: loss=0.2173, val

[I 2026-05-26 21:35:00,910] Trial 1 finished with value: 0.749362932302185 and parameters: {'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.06218303726628978, 'ft_epochs': 7, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}. Best is trial 0 with value: 0.7595277674408643.


...

FT epoch 6/6: loss=0.0789, val_f1=0.9126
Fold 15 subject 17 {'accuracy': 0.6438356164383562, 'f1': 0.0, 'balanced_accuracy': 0.46078431372549017, 'auroc': 0.06506238859180036, 'subject': 17}
SSL epoch 1/4: 2.9511
SSL epoch 2/4: 2.3888
SSL epoch 3/4: 2.1484
SSL epoch 4/4: 2.0576
FT epoch 1/7: loss=0.3913, val_f1=0.8043
FT epoch 2/7: loss=0.2351, val_f1=0.8276
FT epoch 3/7: loss=0.1830, val_f1=0.8387
FT epoch 4/7: loss=0.1684, val_f1=0.8539
FT epoch 5/7: loss=0.1558, val_f1=0.8736
FT epoch 6/7: loss=0.1692, val_f1=0.8696
FT epoch 7/7: loss=0.1245, val_f1=0.8696
Fold 1 subject 2 {'accuracy': 0.9402985074626866, 'f1': 0.9090909090909091, 'balanced_accuracy': 0.9574468085106382, 'auroc': 0.9840425531914894, 'subject': 2}
SSL epoch 1/4: 3.0068
SSL epoch 2/4: 2.3891
SSL epoch 3/4: 2.2759
SSL epoch 4/4: 2.2530
FT epoch 1/7: loss=0.3238, val_f1=0.8125
FT epoch 2/7: loss=0.2738, val_f1=0.8431
FT epoch 3/7: loss=0.2152, val_f1=0.8085
FT epoch 4/7: loss=0.1635, val_f1=0.8542
FT epoch 5/7: loss=0.1

[I 2026-05-26 21:37:34,068] Trial 3 finished with value: 0.7544987160421247 and parameters: {'patch_len': 8, 'emb_dim': 256, 'depth': 6, 'dropout': 0.20593139006678607, 'ft_epochs': 7, 'ssl_epochs': 4, 'batch_size': 16, 'use_ssl': True}. Best is trial 0 with value: 0.7595277674408643.


FT epoch 7/7: loss=0.1027, val_f1=0.9383
Fold 15 subject 17 {'accuracy': 0.4520547945205479, 'f1': 0.0, 'balanced_accuracy': 0.3235294117647059, 'auroc': 0.0882352941176471, 'subject': 17}
SSL epoch 1/7: 2.6347
SSL epoch 2/7: 2.2718
SSL epoch 3/7: 2.1747
SSL epoch 4/7: 2.0817
SSL epoch 5/7: 2.0309
SSL epoch 6/7: 1.9108
SSL epoch 7/7: 1.8258
FT epoch 1/7: loss=0.3069, val_f1=0.8506
FT epoch 2/7: loss=0.2114, val_f1=0.8706
FT epoch 3/7: loss=0.1724, val_f1=0.9070
FT epoch 4/7: loss=0.1469, val_f1=0.8478
FT epoch 5/7: loss=0.1270, val_f1=0.9302
FT epoch 6/7: loss=0.1052, val_f1=0.9195
FT epoch 7/7: loss=0.0660, val_f1=0.8916
Fold 1 subject 2 {'accuracy': 1.0, 'f1': 1.0, 'balanced_accuracy': 1.0, 'auroc': 1.0, 'subject': 2}
SSL epoch 1/7: 2.7587
SSL epoch 2/7: 2.3050
SSL epoch 3/7: 2.1797
SSL epoch 4/7: 2.1203
SSL epoch 5/7: 2.0388
SSL epoch 6/7: 1.9784
SSL epoch 7/7: 1.8880
FT epoch 1/7: loss=0.3347, val_f1=0.8642
FT epoch 2/7: loss=0.2058, val_f1=0.8333
FT epoch 3/7: loss=0.1619, val_f1=

[I 2026-05-26 21:38:52,806] Trial 4 finished with value: 0.7422640528381366 and parameters: {'patch_len': 16, 'emb_dim': 256, 'depth': 4, 'dropout': 0.17238012540141584, 'ft_epochs': 7, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}. Best is trial 0 with value: 0.7595277674408643.


FT epoch 7/7: loss=0.0284, val_f1=0.9067
Fold 15 subject 17 {'accuracy': 0.6986301369863014, 'f1': 0.0, 'balanced_accuracy': 0.5, 'auroc': 0.13458110516934046, 'subject': 17}

OPTUNA FINALIZADO
Best F1: 0.7595
Best params: {'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.05587873967732661, 'ft_epochs': 6, 'ssl_epochs': 10, 'batch_size': 32, 'use_ssl': True}
Resultados guardados en: /home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/wesad_optuna_60seg.json

Mejor F1: 0.7595277674408643
Mejores parámetros: {'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.05587873967732661, 'ft_epochs': 6, 'ssl_epochs': 10, 'batch_size': 32, 'use_ssl': True}


In [10]:
# 1) Supervised only

import json

path = f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/wesad_optuna_{WINDOW_SECONDS}seg.json"

with open(path, "r") as f:
    data = json.load(f)

best_params = data["best_params"]
print(best_params)

res_sup = run_loso(
X, y, groups,
use_ssl=False,
ssl_epochs=best_params["ssl_epochs"],
ft_epochs=best_params["ft_epochs"],
emb_dim=best_params["emb_dim"],
depth=best_params["depth"],
dropout=best_params["dropout"]
)

summarize("Supervised only", res_sup,f"resultados_Supervisado_WESAD_{WINDOW_SECONDS}seg")

{'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.05587873967732661, 'ft_epochs': 6, 'ssl_epochs': 10, 'batch_size': 32, 'use_ssl': True}
FT epoch 1/6: loss=0.3885, val_f1=0.8395
FT epoch 2/6: loss=0.2669, val_f1=0.8608
FT epoch 3/6: loss=0.2727, val_f1=0.8462
FT epoch 4/6: loss=0.2567, val_f1=0.8608
FT epoch 5/6: loss=0.2235, val_f1=0.8378
FT epoch 6/6: loss=0.2165, val_f1=0.8611
Fold 1 subject 2 {'accuracy': 1.0, 'f1': 1.0, 'balanced_accuracy': 1.0, 'auroc': 1.0, 'subject': 2}
FT epoch 1/6: loss=0.3111, val_f1=0.8916
FT epoch 2/6: loss=0.2488, val_f1=0.8966
FT epoch 3/6: loss=0.2070, val_f1=0.8864
FT epoch 4/6: loss=0.2002, val_f1=0.8941
FT epoch 5/6: loss=0.1891, val_f1=0.8837
FT epoch 6/6: loss=0.1852, val_f1=0.9048
Fold 2 subject 3 {'accuracy': 0.7352941176470589, 'f1': 0.6538461538461539, 'balanced_accuracy': 0.76875, 'auroc': 0.84375, 'subject': 3}
FT epoch 1/6: loss=0.3031, val_f1=0.8119
FT epoch 2/6: loss=0.2443, val_f1=0.8269
FT epoch 3/6: loss=0.2545, val_f1=0.8163


{'accuracy': {'mean': 0.8716948015727397,
  'std': 0.1868413893797232,
  'ci95_low': 0.7682254889339571,
  'ci95_high': 0.9751641142115223,
  'n': 15},
 'f1': {'mean': 0.7676841805325209,
  'std': 0.3589186514170232,
  'ci95_low': 0.5689216564544995,
  'ci95_high': 0.9664467046105422,
  'n': 15},
 'balanced_accuracy': {'mean': 0.8490098312052095,
  'std': 0.2351011344065916,
  'ci95_low': 0.7188151625699312,
  'ci95_high': 0.9792044998404877,
  'n': 15},
 'auroc': {'mean': 0.8888272733335758,
  'std': 0.24281356723270153,
  'ci95_low': 0.7543616017586406,
  'ci95_high': 1.023292944908511,
  'n': 15}}

In [11]:
# 2) SSL + supervised

import json

path = f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/wesad_optuna_{WINDOW_SECONDS}seg.json"

with open(path, "r") as f:
    data = json.load(f)

best_params = data["best_params"]
print(best_params)

res_ssl = run_loso(
X, y, groups,
use_ssl=best_params["use_ssl"],
ssl_epochs=best_params["ssl_epochs"],
ft_epochs=best_params["ft_epochs"],
emb_dim=best_params["emb_dim"],
depth=best_params["depth"],
dropout=best_params["dropout"]
)

summarize("SSL + supervised", res_ssl,f"resultados_SSL_WESAD_{WINDOW_SECONDS}seg")

{'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.05587873967732661, 'ft_epochs': 6, 'ssl_epochs': 10, 'batch_size': 32, 'use_ssl': True}
SSL epoch 1/10: 2.8033
SSL epoch 2/10: 2.3017
SSL epoch 3/10: 2.1050
SSL epoch 4/10: 2.0349
SSL epoch 5/10: 1.9426
SSL epoch 6/10: 1.8653
SSL epoch 7/10: 1.7776
SSL epoch 8/10: 1.7322
SSL epoch 9/10: 1.6861
SSL epoch 10/10: 1.6145
FT epoch 1/6: loss=0.3608, val_f1=0.7865
FT epoch 2/6: loss=0.2204, val_f1=0.8043
FT epoch 3/6: loss=0.1999, val_f1=0.8140
FT epoch 4/6: loss=0.1819, val_f1=0.8043
FT epoch 5/6: loss=0.1717, val_f1=0.8537
FT epoch 6/6: loss=0.1735, val_f1=0.8642
Fold 1 subject 2 {'accuracy': 1.0, 'f1': 1.0, 'balanced_accuracy': 1.0, 'auroc': 1.0, 'subject': 2}
SSL epoch 1/10: 2.7228
SSL epoch 2/10: 2.2658
SSL epoch 3/10: 2.1245
SSL epoch 4/10: 2.0498
SSL epoch 5/10: 1.9118
SSL epoch 6/10: 1.8529
SSL epoch 7/10: 1.7574
SSL epoch 8/10: 1.6894
SSL epoch 9/10: 1.6394
SSL epoch 10/10: 1.5921
FT epoch 1/6: loss=0.3598, val_f1=0.8810
FT e

{'accuracy': {'mean': 0.8790955570171558,
  'std': 0.17279035531638667,
  'ci95_low': 0.783407447682545,
  'ci95_high': 0.9747836663517666,
  'n': 15},
 'f1': {'mean': 0.7788261464162424,
  'std': 0.33846844641017754,
  'ci95_low': 0.591388568392225,
  'ci95_high': 0.9662637244402598,
  'n': 15},
 'balanced_accuracy': {'mean': 0.857852638024907,
  'std': 0.21969217649419306,
  'ci95_low': 0.7361911658562614,
  'ci95_high': 0.9795141101935525,
  'n': 15},
 'auroc': {'mean': 0.8855129385087367,
  'std': 0.23663515804913,
  'ci95_low': 0.7544687558958969,
  'ci95_high': 1.0165571211215765,
  'n': 15}}

### Mejores Hiperparámetros STRESS-ID

In [80]:
# EJECUTAR OPTUNA
print("INICIANDO OPTUNA...")
study_optuna_Stress_ID = run_optuna_search(
    X1, y1, groups_1,
    n_trials=5,
    study_name=f"Stress_ID_optuna_{WINDOW_SECONDS}seg"
)

print("\nMejor F1:", study_optuna_Stress_ID.best_value)
print("Mejores parámetros:", study_optuna_Stress_ID.best_params)

[I 2026-05-26 21:42:13,325] A new study created in memory with name: Stress_ID_optuna_60seg


INICIANDO OPTUNA...
SSL epoch 1/10: 2.3655
SSL epoch 2/10: 1.7287
SSL epoch 3/10: 1.5435
SSL epoch 4/10: 1.3856
SSL epoch 5/10: 1.3171
SSL epoch 6/10: 1.2557
SSL epoch 7/10: 1.2394
SSL epoch 8/10: 1.1858
SSL epoch 9/10: 1.1466
SSL epoch 10/10: 1.1462
FT epoch 1/6: loss=0.6850, val_f1=0.7365
FT epoch 2/6: loss=0.6530, val_f1=0.7329
FT epoch 3/6: loss=0.6528, val_f1=0.7000
FT epoch 4/6: loss=0.6462, val_f1=0.6844
FT epoch 5/6: loss=0.6305, val_f1=0.7078
FT epoch 6/6: loss=0.6227, val_f1=0.6690
Fold 1 subject 2 {'accuracy': 0.58, 'f1': 0.72, 'balanced_accuracy': 0.48811544991511036, 'auroc': 0.7062818336162988, 'subject': 2}
SSL epoch 1/10: 2.3297
SSL epoch 2/10: 1.7743
SSL epoch 3/10: 1.5181
SSL epoch 4/10: 1.4290
SSL epoch 5/10: 1.3476
SSL epoch 6/10: 1.2714
SSL epoch 7/10: 1.2070
SSL epoch 8/10: 1.1977
SSL epoch 9/10: 1.1467
SSL epoch 10/10: 1.1479
FT epoch 1/6: loss=0.6924, val_f1=0.7546
FT epoch 2/6: loss=0.6563, val_f1=0.7362
FT epoch 3/6: loss=0.6465, val_f1=0.7554
FT epoch 4/6: lo

[I 2026-05-26 21:46:06,947] Trial 0 finished with value: 0.7141182030404137 and parameters: {'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.05587873967732661, 'ft_epochs': 6, 'ssl_epochs': 10, 'batch_size': 32, 'use_ssl': True}. Best is trial 0 with value: 0.7141182030404137.


FT epoch 6/6: loss=0.6252, val_f1=0.7089
Fold 33 subject 35 {'accuracy': 0.6122448979591837, 'f1': 0.7466666666666667, 'balanced_accuracy': 0.507168458781362, 'auroc': 0.4354838709677419, 'subject': 35}
SSL epoch 1/7: 2.5051
SSL epoch 2/7: 1.7465
SSL epoch 3/7: 1.5138
SSL epoch 4/7: 1.4327
SSL epoch 5/7: 1.3428
SSL epoch 6/7: 1.2816
SSL epoch 7/7: 1.2786
FT epoch 1/7: loss=0.6965, val_f1=0.6972
FT epoch 2/7: loss=0.6647, val_f1=0.7412
FT epoch 3/7: loss=0.6543, val_f1=0.7622
FT epoch 4/7: loss=0.6535, val_f1=0.7530
FT epoch 5/7: loss=0.6511, val_f1=0.7331
FT epoch 6/7: loss=0.6402, val_f1=0.7631
FT epoch 7/7: loss=0.6347, val_f1=0.7195
Fold 1 subject 2 {'accuracy': 0.64, 'f1': 0.7352941176470589, 'balanced_accuracy': 0.5874363327674024, 'auroc': 0.6366723259762309, 'subject': 2}
SSL epoch 1/7: 2.4319
SSL epoch 2/7: 1.8005
SSL epoch 3/7: 1.5575
SSL epoch 4/7: 1.4183
SSL epoch 5/7: 1.3723
SSL epoch 6/7: 1.3241
SSL epoch 7/7: 1.2543
FT epoch 1/7: loss=0.6767, val_f1=0.7121
FT epoch 2/7: l

[I 2026-05-26 21:49:18,625] Trial 1 finished with value: 0.7167658612518916 and parameters: {'patch_len': 16, 'emb_dim': 128, 'depth': 3, 'dropout': 0.06218303726628978, 'ft_epochs': 7, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}. Best is trial 1 with value: 0.7167658612518916.


...

FT epoch 6/6: loss=0.6562, val_f1=0.7093
Fold 33 subject 35 {'accuracy': 0.6530612244897959, 'f1': 0.7671232876712328, 'balanced_accuracy': 0.5627240143369175, 'auroc': 0.439068100358423, 'subject': 35}
SSL epoch 1/4: 2.7739
SSL epoch 2/4: 2.3686
SSL epoch 3/4: 2.2963
SSL epoch 4/4: 2.0942
FT epoch 1/7: loss=0.7045, val_f1=0.6916
FT epoch 2/7: loss=0.6825, val_f1=0.6949
FT epoch 3/7: loss=0.6747, val_f1=0.6951
FT epoch 4/7: loss=0.6687, val_f1=0.6580
FT epoch 5/7: loss=0.6662, val_f1=0.6959
FT epoch 6/7: loss=0.6649, val_f1=0.6962
FT epoch 7/7: loss=0.6668, val_f1=0.6810
Fold 1 subject 2 {'accuracy': 0.64, 'f1': 0.7567567567567568, 'balanced_accuracy': 0.5568760611205432, 'auroc': 0.6451612903225807, 'subject': 2}
SSL epoch 1/4: 2.9795
SSL epoch 2/4: 2.3235
SSL epoch 3/4: 2.0627
SSL epoch 4/4: 1.9515
FT epoch 1/7: loss=0.6946, val_f1=0.7637
FT epoch 2/7: loss=0.6670, val_f1=0.7701
FT epoch 3/7: loss=0.6720, val_f1=0.7221
FT epoch 4/7: loss=0.6626, val_f1=0.7765
FT epoch 5/7: loss=0.656

[I 2026-05-26 21:58:06,111] Trial 3 finished with value: 0.7252783692866015 and parameters: {'patch_len': 8, 'emb_dim': 256, 'depth': 6, 'dropout': 0.20593139006678607, 'ft_epochs': 7, 'ssl_epochs': 4, 'batch_size': 16, 'use_ssl': True}. Best is trial 2 with value: 0.7262089293189584.


FT epoch 7/7: loss=0.6484, val_f1=0.7095
Fold 33 subject 35 {'accuracy': 0.6326530612244898, 'f1': 0.7692307692307693, 'balanced_accuracy': 0.5116487455197133, 'auroc': 0.514336917562724, 'subject': 35}
SSL epoch 1/7: 2.6095
SSL epoch 2/7: 2.0159
SSL epoch 3/7: 1.7592
SSL epoch 4/7: 1.6621
SSL epoch 5/7: 1.5561
SSL epoch 6/7: 1.5333
SSL epoch 7/7: 1.5056
FT epoch 1/7: loss=0.6939, val_f1=0.6757
FT epoch 2/7: loss=0.6650, val_f1=0.6884
FT epoch 3/7: loss=0.6547, val_f1=0.7108
FT epoch 4/7: loss=0.6499, val_f1=0.6854
FT epoch 5/7: loss=0.6489, val_f1=0.6535
FT epoch 6/7: loss=0.6436, val_f1=0.6142
FT epoch 7/7: loss=0.6258, val_f1=0.6754
Fold 1 subject 2 {'accuracy': 0.62, 'f1': 0.7466666666666667, 'balanced_accuracy': 0.530560271646859, 'auroc': 0.699490662139219, 'subject': 2}
SSL epoch 1/7: 2.6171
SSL epoch 2/7: 2.0554
SSL epoch 3/7: 1.8077
SSL epoch 4/7: 1.7832
SSL epoch 5/7: 1.6736
SSL epoch 6/7: 1.5230
SSL epoch 7/7: 1.4844
FT epoch 1/7: loss=0.6993, val_f1=0.7561
FT epoch 2/7: los

[I 2026-05-26 22:02:36,961] Trial 4 finished with value: 0.7256427734276568 and parameters: {'patch_len': 16, 'emb_dim': 256, 'depth': 4, 'dropout': 0.17238012540141584, 'ft_epochs': 7, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}. Best is trial 2 with value: 0.7262089293189584.


FT epoch 7/7: loss=0.6310, val_f1=0.6962
Fold 33 subject 35 {'accuracy': 0.6530612244897959, 'f1': 0.7792207792207793, 'balanced_accuracy': 0.5394265232974911, 'auroc': 0.5286738351254481, 'subject': 35}

OPTUNA FINALIZADO
Best F1: 0.7262
Best params: {'patch_len': 8, 'emb_dim': 256, 'depth': 4, 'dropout': 0.20059047112039313, 'ft_epochs': 6, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}
Resultados guardados en: /home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/Stress_ID_optuna_60seg.json

Mejor F1: 0.7262089293189584
Mejores parámetros: {'patch_len': 8, 'emb_dim': 256, 'depth': 4, 'dropout': 0.20059047112039313, 'ft_epochs': 6, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}


In [12]:
# 1) Supervised only

import json

path = f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/Stress_ID_optuna_{WINDOW_SECONDS}seg.json"

with open(path, "r") as f:
    data = json.load(f)

best_params = data["best_params"]
print(best_params)

res_sup_1 = run_loso(
X1, y1, groups_1,
use_ssl=False,
ssl_epochs=best_params["ssl_epochs"],
ft_epochs=best_params["ft_epochs"],
emb_dim=best_params["emb_dim"],
depth=best_params["depth"],
dropout=best_params["dropout"]
)

summarize("Supervised only", res_sup_1,f"resultados_Supervisado_StressID_{WINDOW_SECONDS}seg")

{'patch_len': 8, 'emb_dim': 256, 'depth': 4, 'dropout': 0.20059047112039313, 'ft_epochs': 6, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}
FT epoch 1/6: loss=0.4708, val_f1=0.8571
FT epoch 2/6: loss=0.3880, val_f1=0.8643
FT epoch 3/6: loss=0.3831, val_f1=0.8621
FT epoch 4/6: loss=0.3658, val_f1=0.8551
FT epoch 5/6: loss=0.3647, val_f1=0.8633
FT epoch 6/6: loss=0.3675, val_f1=0.8721
Fold 1 subject 2 {'accuracy': 0.96, 'f1': 0.967741935483871, 'balanced_accuracy': 0.9575551782682512, 'auroc': 0.9983022071307301, 'subject': 2}
FT epoch 1/6: loss=0.4571, val_f1=0.8487
FT epoch 2/6: loss=0.3822, val_f1=0.8199
FT epoch 3/6: loss=0.3755, val_f1=0.8534
FT epoch 4/6: loss=0.3690, val_f1=0.8452
FT epoch 5/6: loss=0.3664, val_f1=0.8217
FT epoch 6/6: loss=0.3517, val_f1=0.8443
Fold 2 subject 3 {'accuracy': 0.9285714285714286, 'f1': 0.9387755102040817, 'balanced_accuracy': 0.9305882352941177, 'auroc': 0.9858823529411764, 'subject': 3}
FT epoch 1/6: loss=0.4140, val_f1=0.8352
FT epoch 2/6: los

{'accuracy': {'mean': 0.8372822628673533,
  'std': 0.16827088684387545,
  'ci95_low': 0.7776160008321652,
  'ci95_high': 0.8969485249025414,
  'n': 33},
 'f1': {'mean': 0.8660619346860782,
  'std': 0.13842300696689508,
  'ci95_low': 0.8169792706537169,
  'ci95_high': 0.9151445987184395,
  'n': 33},
 'balanced_accuracy': {'mean': 0.8313952565648548,
  'std': 0.18126977757131113,
  'ci95_low': 0.767119788297637,
  'ci95_high': 0.8956707248320726,
  'n': 33},
 'auroc': {'mean': 0.9082579474134433,
  'std': 0.15252043787664485,
  'ci95_low': 0.8541765517379516,
  'ci95_high': 0.9623393430889351,
  'n': 33}}

In [13]:
# 2) SSL + supervised

import json

path = f"/home/leoisidro/CICLOS/X/PFC_II/PFC2/results_optuna/Stress_ID_optuna_{WINDOW_SECONDS}seg.json"

with open(path, "r") as f:
    data = json.load(f)

best_params = data["best_params"]
print(best_params)

res_ssl_1 = run_loso(
X1, y1, groups_1,
use_ssl=best_params["use_ssl"],
ssl_epochs=best_params["ssl_epochs"],
ft_epochs=best_params["ft_epochs"],
emb_dim=best_params["emb_dim"],
depth=best_params["depth"],
dropout=best_params["dropout"]
)
summarize("SSL + supervised", res_ssl_1,f"resultados_SSL_StressID_{WINDOW_SECONDS}seg")

{'patch_len': 8, 'emb_dim': 256, 'depth': 4, 'dropout': 0.20059047112039313, 'ft_epochs': 6, 'ssl_epochs': 7, 'batch_size': 16, 'use_ssl': True}
SSL epoch 1/7: 2.5854
SSL epoch 2/7: 2.1622
SSL epoch 3/7: 2.0197
SSL epoch 4/7: 1.9053
SSL epoch 5/7: 1.8261
SSL epoch 6/7: 1.7234
SSL epoch 7/7: 1.6969
FT epoch 1/6: loss=0.4504, val_f1=0.8881
FT epoch 2/6: loss=0.3636, val_f1=0.8485
FT epoch 3/6: loss=0.3724, val_f1=0.7773
FT epoch 4/6: loss=0.3380, val_f1=0.8571
FT epoch 5/6: loss=0.3206, val_f1=0.8746
FT epoch 6/6: loss=0.3152, val_f1=0.8540
Fold 1 subject 2 {'accuracy': 1.0, 'f1': 1.0, 'balanced_accuracy': 1.0, 'auroc': 1.0, 'subject': 2}
SSL epoch 1/7: 2.6353
SSL epoch 2/7: 2.1850
SSL epoch 3/7: 2.1495
SSL epoch 4/7: 2.1208
SSL epoch 5/7: 2.0269
SSL epoch 6/7: 2.0196
SSL epoch 7/7: 1.8808
FT epoch 1/6: loss=0.4102, val_f1=0.8392
FT epoch 2/6: loss=0.3598, val_f1=0.8542
FT epoch 3/6: loss=0.3539, val_f1=0.8581
FT epoch 4/6: loss=0.3425, val_f1=0.8632
FT epoch 5/6: loss=0.3381, val_f1=0.8

{'accuracy': {'mean': 0.8454959530964739,
  'std': 0.1507735037597964,
  'ci95_low': 0.7920339933241803,
  'ci95_high': 0.8989579128687674,
  'n': 33},
 'f1': {'mean': 0.8716963539899963,
  'std': 0.12753063860815125,
  'ci95_low': 0.8264759558365782,
  'ci95_high': 0.9169167521434145,
  'n': 33},
 'balanced_accuracy': {'mean': 0.8393814742981898,
  'std': 0.15987032649979013,
  'ci95_low': 0.7826939214392221,
  'ci95_high': 0.8960690271571574,
  'n': 33},
 'auroc': {'mean': 0.8968503068311752,
  'std': 0.1549983211272059,
  'ci95_low': 0.8418902919596145,
  'ci95_high': 0.9518103217027358,
  'n': 33}}

### Cruzamos los mejores hiperparámetros entre WESAD y STRESS-ID entrenamos en un dataset y evaluamos en el otro para ver si generalizan

In [14]:
import json
import os

import numpy as np
import torch
from sklearn.model_selection import train_test_split


def load_best_params(study_name):
    path = os.path.join(RESULTS_DIR, f"{study_name}.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["best_params"]


def train_on_source_eval_target(source_name, X_src, y_src, target_name, X_tgt, y_tgt, best_params):
    torch.cuda.empty_cache()

    batch_size = int(best_params.get("batch_size", 64))
    patch_len = int(best_params.get("patch_len", 8))

    X_train, X_val, y_train, y_val = train_test_split(
        X_src,
        y_src,
        test_size=0.15,
        random_state=SEED,
        stratify=y_src,
    )

    model = TimePatchTransformer(
        in_ch=3,
        patch_len=patch_len,
        emb_dim=best_params["emb_dim"],
        depth=best_params["depth"],
        heads=4,
        dropout=best_params["dropout"],
        num_classes=2,
    )

    if best_params.get("use_ssl", False):
        ssl_batch_size = min(max(2, batch_size), len(X_train))
        model = pretrain_ssl(
            model,
            X_train.copy(),
            epochs=best_params["ssl_epochs"],
            batch_size=ssl_batch_size,
        )

    ft_batch_size = min(max(2, batch_size), len(X_train))
    model = finetune(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=best_params["ft_epochs"],
        batch_size=ft_batch_size,
    )

    test_batch_size = min(max(2, batch_size), len(X_tgt))
    probs = predict_probs(model, X_tgt, batch_size=test_batch_size)
    metrics = evaluate_probs(y_tgt, probs)

    print(f"\n{source_name} -> {target_name}")
    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        print(f"{key}: {metrics[key]:.4f}")

    return metrics


wesad_params = load_best_params(f"wesad_optuna_{WINDOW_SECONDS}seg")
stress_params = load_best_params(f"Stress_ID_optuna_{WINDOW_SECONDS}seg")

res_wesad_to_stress = train_on_source_eval_target(
    "WESAD",
    X,
    y,
    "STRESS-ID",
    X1,
    y1,
    wesad_params,
)
summarize(
    "WESAD->STRESS-ID",
    [res_wesad_to_stress],
    f"resultados_Cruzados_WESAD_a_STRESSID_all_{WINDOW_SECONDS}seg",
    append=False,
)

res_stress_to_wesad = train_on_source_eval_target(
    "STRESS-ID",
    X1,
    y1,
    "WESAD",
    X,
    y,
    stress_params,
)
summarize(
    "STRESS-ID->WESAD",
    [res_stress_to_wesad],
    f"resultados_Cruzados_STRESSID_a_WESAD_all_{WINDOW_SECONDS}seg",
    append=False,
)

print("\nCruce train en un dataset / test en el otro finalizado.")

SSL epoch 1/10: 2.0120
SSL epoch 2/10: 1.6468
SSL epoch 3/10: 1.4559
SSL epoch 4/10: 1.4092
SSL epoch 5/10: 1.3213
SSL epoch 6/10: 1.2518
SSL epoch 7/10: 1.2114
SSL epoch 8/10: 1.1594
SSL epoch 9/10: 1.1172
SSL epoch 10/10: 1.0842
FT epoch 1/6: loss=0.2917, val_f1=0.8182
FT epoch 2/6: loss=0.2157, val_f1=0.7907
FT epoch 3/6: loss=0.1946, val_f1=0.8605
FT epoch 4/6: loss=0.1824, val_f1=0.8571
FT epoch 5/6: loss=0.1789, val_f1=0.8632
FT epoch 6/6: loss=0.1570, val_f1=0.8791

WESAD -> STRESS-ID
accuracy: 0.5965
f1: 0.4882
balanced_accuracy: 0.6562
auroc: 0.8517

WESAD->STRESS-ID
  accuracy              : 0.5965 ± nan  [IC95%: (nan, nan)]  n=1
  f1                    : 0.4882 ± nan  [IC95%: (nan, nan)]  n=1
  balanced_accuracy     : 0.6562 ± nan  [IC95%: (nan, nan)]  n=1
  auroc                 : 0.8517 ± nan  [IC95%: (nan, nan)]  n=1


/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


SSL epoch 1/7: 1.3606
SSL epoch 2/7: 1.3083
SSL epoch 3/7: 1.1568
SSL epoch 4/7: 1.1381
SSL epoch 5/7: 1.0985
SSL epoch 6/7: 1.1397
SSL epoch 7/7: 1.0786
FT epoch 1/6: loss=0.4219, val_f1=0.8581
FT epoch 2/6: loss=0.3802, val_f1=0.8561
FT epoch 3/6: loss=0.3673, val_f1=0.8561
FT epoch 4/6: loss=0.3518, val_f1=0.8763
FT epoch 5/6: loss=0.3565, val_f1=0.8639
FT epoch 6/6: loss=0.3460, val_f1=0.8231

STRESS-ID -> WESAD
accuracy: 0.5245
f1: 0.5540
balanced_accuracy: 0.6613
auroc: 0.9073

STRESS-ID->WESAD
  accuracy              : 0.5245 ± nan  [IC95%: (nan, nan)]  n=1
  f1                    : 0.5540 ± nan  [IC95%: (nan, nan)]  n=1
  balanced_accuracy     : 0.6613 ± nan  [IC95%: (nan, nan)]  n=1
  auroc                 : 0.9073 ± nan  [IC95%: (nan, nan)]  n=1

Cruce train en un dataset / test en el otro finalizado.


/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


### Cruzamos mejores hiperparámetros entre WESAD y STRESS-ID entrenamos en el 80% de los dos datasets y evaluamos en el 20% restante para ver si generalizan

In [15]:
import json
import os

import numpy as np
import torch
from sklearn.model_selection import train_test_split


def load_best_params(study_name):
    path = os.path.join(RESULTS_DIR, f"{study_name}.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["best_params"]


def resample_window_to_length(window, target_len):
    window = np.asarray(window, dtype=np.float32)
    if window.shape[0] == target_len:
        return window
    xp = np.linspace(0.0, 1.0, window.shape[0])
    x = np.linspace(0.0, 1.0, target_len)
    resized = [np.interp(x, xp, window[:, channel]) for channel in range(window.shape[1])]
    return np.stack(resized, axis=1).astype(np.float32)


def resize_dataset(X, target_len):
    return np.stack([resample_window_to_length(sample, target_len) for sample in X], axis=0)


def train_combined_and_eval(combo_name, X_src_a, y_src_a, X_src_b, y_src_b, best_params):
    torch.cuda.empty_cache()

    common_len = min(X_src_a.shape[1], X_src_b.shape[1])
    X_src_a = resize_dataset(X_src_a, common_len)
    X_src_b = resize_dataset(X_src_b, common_len)

    X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(
        X_src_a,
        y_src_a,
        test_size=0.2,
        random_state=SEED,
        stratify=y_src_a,
    )
    X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
        X_src_b,
        y_src_b,
        test_size=0.2,
        random_state=SEED,
        stratify=y_src_b,
    )

    X_train = np.concatenate([X_a_train, X_b_train], axis=0)
    y_train = np.concatenate([y_a_train, y_b_train], axis=0)

    X_val_train, X_val, y_val_train, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train,
    )

    batch_size = int(best_params.get("batch_size", 64))
    patch_len = int(best_params.get("patch_len", 8))

    model = TimePatchTransformer(
        in_ch=3,
        patch_len=patch_len,
        emb_dim=best_params["emb_dim"],
        depth=best_params["depth"],
        heads=4,
        dropout=best_params["dropout"],
        num_classes=2,
    )

    if best_params.get("use_ssl", False):
        ssl_batch_size = min(max(2, batch_size), len(X_val_train))
        model = pretrain_ssl(
            model,
            X_val_train.copy(),
            epochs=best_params["ssl_epochs"],
            batch_size=ssl_batch_size,
        )

    ft_batch_size = min(max(2, batch_size), len(X_val_train))
    model = finetune(
        model,
        X_val_train,
        y_val_train,
        X_val,
        y_val,
        epochs=best_params["ft_epochs"],
        batch_size=ft_batch_size,
    )

    test_a_batch = min(max(2, batch_size), len(X_a_test))
    test_b_batch = min(max(2, batch_size), len(X_b_test))

    metrics_a = evaluate_probs(y_a_test, predict_probs(model, X_a_test, batch_size=test_a_batch))
    metrics_b = evaluate_probs(y_b_test, predict_probs(model, X_b_test, batch_size=test_b_batch))

    print(f"\n{combo_name}")
    print(f"Common window length: {common_len}")
    print("Test en el 20% de WESAD")
    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        print(f"{key}: {metrics_a[key]:.4f}")
    print("Test en el 20% de STRESS-ID")
    for key in ["accuracy", "f1", "balanced_accuracy", "auroc"]:
        print(f"{key}: {metrics_b[key]:.4f}")

    return metrics_a, metrics_b


wesad_params = load_best_params(f"wesad_optuna_{WINDOW_SECONDS}seg")
stress_params = load_best_params(f"Stress_ID_optuna_{WINDOW_SECONDS}seg")

metrics_wesad_params = train_combined_and_eval(
    "Entrenamiento combinado usando hiperparámetros de WESAD",
    X,
    y,
    X1,
    y1,
    wesad_params,
)
summarize(
    "Combinado->WESAD params/test WESAD",
    [metrics_wesad_params[0]],
    f"resultados_Combo_WESADparams_test_WESAD_{WINDOW_SECONDS}seg",
    append=False,
)
summarize(
    "Combinado->WESAD params/test STRESS-ID",
    [metrics_wesad_params[1]],
    f"resultados_Combo_WESADparams_test_STRESSID_{WINDOW_SECONDS}seg",
    append=False,
)

metrics_stress_params = train_combined_and_eval(
    "Entrenamiento combinado usando hiperparámetros de STRESS-ID",
    X,
    y,
    X1,
    y1,
    stress_params,
)
summarize(
    "Combinado->STRESS params/test WESAD",
    [metrics_stress_params[0]],
    f"resultados_Combo_STRESSparams_test_WESAD_{WINDOW_SECONDS}seg",
    append=False,
)
summarize(
    "Combinado->STRESS params/test STRESS-ID",
    [metrics_stress_params[1]],
    f"resultados_Combo_STRESSparams_test_STRESSID_{WINDOW_SECONDS}seg",
    append=False,
)

print("\nExperimento combinado 80/20 finalizado.")

SSL epoch 1/10: 1.7105
SSL epoch 2/10: 1.2859
SSL epoch 3/10: 1.1463
SSL epoch 4/10: 1.0585
SSL epoch 5/10: 0.9908
SSL epoch 6/10: 0.9051
SSL epoch 7/10: 0.8689
SSL epoch 8/10: 0.8406
SSL epoch 9/10: 0.8130
SSL epoch 10/10: 0.7757
FT epoch 1/6: loss=0.4317, val_f1=0.8391
FT epoch 2/6: loss=0.3365, val_f1=0.8656
FT epoch 3/6: loss=0.3021, val_f1=0.8816
FT epoch 4/6: loss=0.2921, val_f1=0.8961
FT epoch 5/6: loss=0.2543, val_f1=0.8721
FT epoch 6/6: loss=0.2292, val_f1=0.8889

Entrenamiento combinado usando hiperparámetros de WESAD
Common window length: 240
Test en el 20% de WESAD
accuracy: 0.8962
f1: 0.8429
balanced_accuracy: 0.9079
auroc: 0.9692
Test en el 20% de STRESS-ID
accuracy: 0.8241
f1: 0.8455
balanced_accuracy: 0.8257
auroc: 0.9130

Combinado->WESAD params/test WESAD
  accuracy              : 0.8962 ± nan  [IC95%: (nan, nan)]  n=1
  f1                    : 0.8429 ± nan  [IC95%: (nan, nan)]  n=1
  balanced_accuracy     : 0.9079 ± nan  [IC95%: (nan, nan)]  n=1
  auroc              

/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


SSL epoch 1/7: 1.3542
SSL epoch 2/7: 1.2494
SSL epoch 3/7: 1.0480
SSL epoch 4/7: 1.1295
SSL epoch 5/7: 1.0921
SSL epoch 6/7: 1.1205
SSL epoch 7/7: 1.0453
FT epoch 1/6: loss=0.4799, val_f1=0.8100
FT epoch 2/6: loss=0.4180, val_f1=0.8323
FT epoch 3/6: loss=0.4016, val_f1=0.8339
FT epoch 4/6: loss=0.3977, val_f1=0.8129
FT epoch 5/6: loss=0.3710, val_f1=0.8318
FT epoch 6/6: loss=0.3604, val_f1=0.8276

Entrenamiento combinado usando hiperparámetros de STRESS-ID
Common window length: 240
Test en el 20% de WESAD
accuracy: 0.8726
f1: 0.8163
balanced_accuracy: 0.8957
auroc: 0.9576
Test en el 20% de STRESS-ID
accuracy: 0.8302
f1: 0.8541
balanced_accuracy: 0.8275
auroc: 0.8702

Combinado->STRESS params/test WESAD
  accuracy              : 0.8726 ± nan  [IC95%: (nan, nan)]  n=1
  f1                    : 0.8163 ± nan  [IC95%: (nan, nan)]  n=1
  balanced_accuracy     : 0.8957 ± nan  [IC95%: (nan, nan)]  n=1
  auroc                 : 0.9576 ± nan  [IC95%: (nan, nan)]  n=1

Combinado->STRESS params/te

/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/leoisidro/miniconda3/envs/spark310/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## Análisis Estadístico: Wilcoxon SSL vs Supervisado Puro

Prueba de Wilcoxon **pareada** entre el modelo SSL y el modelo supervisado puro para WESAD y Stress-ID, sobre F1, Balanced Accuracy y AUROC.

Incluye **rank-biserial correlation** (tamaño del efecto) y **corrección de Bonferroni** para comparaciones múltiples.

> Referencia: `Wilcoxon_dos_modelos_dos_datasets.ipynb`

> **Prerequisito**: ejecutar celdas 28–29 (WESAD) y 32–33 (Stress-ID) para tener `res_sup`, `res_ssl`, `res_sup_1`, `res_ssl_1`.

In [16]:
# ============================================================
# ANÁLISIS ESTADÍSTICO: Wilcoxon SSL vs Supervisado Puro
# ============================================================
from scipy.stats import wilcoxon as _wilcoxon
from scipy.stats import rankdata as _rankdata
import pandas as pd


def rank_biserial(a, b):
    """Rank-biserial correlation como tamaño del efecto para Wilcoxon.

    Interpretación del |valor|:
      < 0.10 → trivial  |  0.10–0.30 → pequeño
      0.30–0.50 → moderado  |  > 0.50 → grande
    """
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[d != 0]
    if len(d) == 0:
        return 0.0
    ranks   = _rankdata(np.abs(d))
    r_plus  = float(ranks[d > 0].sum())
    r_minus = float(ranks[d < 0].sum())
    return (r_plus - r_minus) / (r_plus + r_minus)


def wilcoxon_compare(results_a, results_b, dataset_name, metric="f1", alpha=0.05):
    """Prueba de Wilcoxon pareada.

    Ambas listas deben cubrir los mismos sujetos (se ordenan por subject ID).
    """
    ra = sorted(results_a, key=lambda r: r["subject"])
    rb = sorted(results_b, key=lambda r: r["subject"])
    a_vals = np.array([r[metric] for r in ra], float)
    b_vals = np.array([r[metric] for r in rb], float)

    stat, p = _wilcoxon(a_vals, b_vals, alternative="two-sided")
    rb_c    = rank_biserial(a_vals, b_vals)
    diffs   = a_vals - b_vals

    print(f"\n{'='*60}")
    print(f"  {dataset_name}  |  Métrica: {metric.upper()}")
    print(f"{'='*60}")
    print(f"  SSL  (A) — media: {a_vals.mean():.4f}  std: {a_vals.std(ddof=1):.4f}")
    print(f"  Sup  (B) — media: {b_vals.mean():.4f}  std: {b_vals.std(ddof=1):.4f}")
    print(f"  Δ A-B    — media: {diffs.mean():.4f}   mediana: {np.median(diffs):.4f}")
    print(f"  W={stat:.2f}   p={p:.6f}   r_rb={rb_c:.3f}")
    sig = "✅ SIGNIFICATIVO" if p < alpha else "⚠️  NO significativo"
    print(f"  {sig}  (α={alpha})")
    return {
        "stat": float(stat), "p": float(p),
        "rank_biserial": float(rb_c),
        "mean_a": float(a_vals.mean()), "mean_b": float(b_vals.mean()),
    }


# --- Wilcoxon por dataset y métrica ---
# Requiere: res_sup, res_ssl (WESAD) y res_sup_1, res_ssl_1 (Stress-ID)
# generados por las celdas 28-29 y 32-33 respectivamente.

_metrics_to_test = ["f1", "balanced_accuracy", "auroc"]
wilcoxon_rows = []

print("\n" + "="*70)
print("WILCOXON: SSL vs Supervisado — WESAD + Stress-ID")
print("="*70)

for _metric in _metrics_to_test:
    for _ds_name, _res_a, _res_b in [
        ("WESAD",     res_ssl,   res_sup),
        ("Stress-ID", res_ssl_1, res_sup_1),
    ]:
        _r = wilcoxon_compare(_res_a, _res_b, _ds_name, _metric)
        wilcoxon_rows.append({
            "Dataset":     _ds_name,
            "Métrica":     _metric,
            "SSL media":   round(_r["mean_a"], 4),
            "Sup media":   round(_r["mean_b"], 4),
            "Δ A-B":       round(_r["mean_a"] - _r["mean_b"], 4),
            "W":           round(_r["stat"], 2),
            "p-value":     round(_r["p"], 6),
            "r_rb":        round(_r["rank_biserial"], 3),
            "Sig (α=0.05)":_r["p"] < 0.05,
        })

wilcoxon_df = pd.DataFrame(wilcoxon_rows)

# Corrección Bonferroni
_n_tests    = len(wilcoxon_rows)
_alpha_bonf = 0.05 / _n_tests
wilcoxon_df["Sig (Bonferroni)"] = wilcoxon_df["p-value"] < _alpha_bonf
print(f"\nCorrección Bonferroni: α_corr = 0.05/{_n_tests} = {_alpha_bonf:.4f}")
display(wilcoxon_df.round(4))



WILCOXON: SSL vs Supervisado — WESAD + Stress-ID

  WESAD  |  Métrica: F1
  SSL  (A) — media: 0.7788  std: 0.3385
  Sup  (B) — media: 0.7677  std: 0.3589
  Δ A-B    — media: 0.0111   mediana: 0.0000
  W=8.00   p=0.310494   r_rb=0.429
  ⚠️  NO significativo  (α=0.05)

  Stress-ID  |  Métrica: F1
  SSL  (A) — media: 0.8717  std: 0.1275
  Sup  (B) — media: 0.8661  std: 0.1384
  Δ A-B    — media: 0.0056   mediana: 0.0000
  W=255.00   p=0.866355   r_rb=0.034
  ⚠️  NO significativo  (α=0.05)

  WESAD  |  Métrica: BALANCED_ACCURACY
  SSL  (A) — media: 0.8579  std: 0.2197
  Sup  (B) — media: 0.8490  std: 0.2351
  Δ A-B    — media: 0.0088   mediana: 0.0000
  W=7.00   p=0.235885   r_rb=0.500
  ⚠️  NO significativo  (α=0.05)

  Stress-ID  |  Métrica: BALANCED_ACCURACY
  SSL  (A) — media: 0.8394  std: 0.1599
  Sup  (B) — media: 0.8314  std: 0.1813
  Δ A-B    — media: 0.0080   mediana: -0.0090
  W=261.00   p=0.955263   r_rb=0.011
  ⚠️  NO significativo  (α=0.05)

  WESAD  |  Métrica: AUROC
  SSL  

,Dataset,Métrica,SSL media,Sup media,Δ A-B,W,p-value,r_rb,Sig (α=0.05),Sig (Bonferroni)
0,WESAD,f1,0.7788,0.7677,0.0111,8.0,0.3105,0.429,False,False
1,Stress-ID,f1,0.8717,0.8661,0.0056,255.0,0.8664,0.034,False,False
2,WESAD,balanced_accuracy,0.8579,0.8490,0.0088,7.0,0.2359,0.500,False,False
3,Stress-ID,balanced_accuracy,0.8394,0.8314,0.0080,261.0,0.9553,0.011,False,False
4,WESAD,auroc,0.8855,0.8888,-0.0033,10.0,0.9165,-0.048,False,False
5,Stress-ID,auroc,0.8969,0.9083,-0.0114,142.0,0.1648,-0.300,False,False


## Experimento 20/80 con Cross-Validation por Paciente

Para **cada sujeto** se realiza un split estratificado 80/20 *dentro de sus propias ventanas*:
- **80%** de ventanas del sujeto → entrenamiento (junto con el 100% del resto)
- **20%** de ventanas del sujeto → test

Esto reemplaza el split global con `train_test_split` por una evaluación **por paciente** que reporta media ± std + IC95% de F1, Balanced Accuracy y AUROC.

Se guarda un **JSON de particiones** con los índices exactos de cada split para total reproducibilidad y auditoría.

In [17]:
# ============================================================
# EXPERIMENTO 20/80 — CROSS-VALIDATION POR PACIENTE
# ============================================================
import os
import json as _json
from datetime import datetime as _datetime


def run_split8020_per_subject(
    X, y, groups,
    use_ssl=False, use_synth=False, ssl_epochs=6, ft_epochs=8,
    synth_ratio=0.5, emb_dim=128, depth=4, heads=4, dropout=0.1,
    test_frac=0.20, seed=42,
    dataset_name="dataset",
    save_dir=None,
):
    """Experimento 20/80 con cross-validation por paciente.

    Para CADA sujeto:
      - 20% de sus ventanas → test  (estratificado por clase)
      - 80% de sus ventanas → train (junto con el 100% del resto de sujetos)

    Parameters
    ----------
    X, y, groups  : arrays de ventanas, etiquetas e IDs de sujeto.
    test_frac     : fracción de ventanas del sujeto → test (default 0.20).
    seed          : semilla para reproducibilidad exacta.
    dataset_name  : str usado en el nombre del archivo JSON.
    save_dir      : directorio donde guardar el JSON de particiones (None → no guarda).

    Returns
    -------
    results       : list of dict — métricas por sujeto.
    partition_log : list of dict — documentación completa de cada partición.
    """
    rng = np.random.default_rng(seed)
    unique_subjects = np.unique(groups)
    results       = []
    partition_log = []

    for subj in unique_subjects:
        subj_idx = np.where(groups == subj)[0]
        rest_idx = np.where(groups != subj)[0]
        y_subj   = y[subj_idx]

        # ── Split estratificado 80/20 dentro del sujeto ────────────────────
        te_local, tr_local           = [], []
        class_dist_train, class_dist_test = {}, {}

        for cls in np.unique(y_subj):
            cls_pos  = np.where(y_subj == cls)[0]
            n_test   = max(1, int(len(cls_pos) * test_frac))
            shuffled = rng.permutation(cls_pos)
            te_local.extend(subj_idx[shuffled[:n_test]].tolist())
            tr_local.extend(subj_idx[shuffled[n_test:]].tolist())
            class_dist_test[int(cls)]  = int(n_test)
            class_dist_train[int(cls)] = int(len(cls_pos) - n_test)

        tr_subj_idx = np.array(tr_local, dtype=int)
        te_idx      = np.array(te_local,  dtype=int)
        tr_idx      = np.concatenate([rest_idx, tr_subj_idx])

        # ── Log de partición ────────────────────────────────────────────────
        partition_log.append({
            "subject":              int(subj),
            "seed":                 int(seed),
            "test_frac":            float(test_frac),
            # índices GLOBALES en el array X (solo ventanas del sujeto)
            "train_window_indices": tr_subj_idx.tolist(),
            "test_window_indices":  te_idx.tolist(),
            "n_train_subject":      int(len(tr_subj_idx)),
            "n_test_subject":       int(len(te_idx)),
            "n_train_total":        int(len(tr_idx)),   # incl. resto de sujetos
            "train_class_dist":     class_dist_train,
            "test_class_dist":      class_dist_test,
            # IDs de todos los sujetos que entran al entrenamiento
            "train_subjects_used":  [int(s) for s in np.unique(groups[rest_idx])],
            "test_subject":         int(subj),
        })

        # ── Entrenamiento ───────────────────────────────────────────────────
        X_train, y_train = X[tr_idx], y[tr_idx]
        X_test,  y_test  = X[te_idx], y[te_idx]

        n_val = max(8, int(0.15 * len(X_train)))
        perm  = rng.permutation(len(X_train))
        X_tr,  y_tr  = X_train[perm[n_val:]], y_train[perm[n_val:]]
        X_val, y_val = X_train[perm[:n_val]], y_train[perm[:n_val]]

        model = TimePatchTransformer(
            in_ch=3, patch_len=8, emb_dim=emb_dim,
            depth=depth, heads=heads, dropout=dropout, num_classes=2)

        if use_ssl:
            model = pretrain_ssl(model, X_train.copy(), epochs=ssl_epochs)

        model = finetune(model, X_tr, y_tr, X_val, y_val, epochs=ft_epochs)
        probs   = predict_probs(model, X_test)
        metrics = evaluate_probs(y_test, probs)
        metrics["subject"] = int(subj)
        results.append(metrics)

        print(f"  Sujeto {subj:3d}  |  "
              f"F1={metrics['f1']:.4f}  "
              f"BA={metrics['balanced_accuracy']:.4f}  "
              f"AUROC={metrics.get('auroc', float('nan')):.4f}  "
              f"[train_subj={len(tr_subj_idx)}, test={len(te_idx)}]")

    # ── Guardar log de particiones en JSON ─────────────────────────────────
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        _ts       = _datetime.now().strftime("%Y%m%d_%H%M%S")
        _log_path = os.path.join(save_dir, f"partition_log_{dataset_name}_{_ts}.json")
        with open(_log_path, "w", encoding="utf-8") as _f:
            _json.dump({
                "dataset":    dataset_name,
                "seed":       int(seed),
                "test_frac":  float(test_frac),
                "n_subjects": int(len(unique_subjects)),
                "partitions": partition_log,
            }, _f, indent=2)
        print(f"\n📄 Log de particiones guardado en: {_log_path}")

    return results, partition_log


# Cargar hiperparámetros óptimos
import json as _j2
_wesad_path = f"{RESULTS_DIR}/wesad_optuna_{WINDOW_SECONDS}seg.json"
_sp_path    = f"{RESULTS_DIR}/Stress_ID_optuna_{WINDOW_SECONDS}seg.json"

with open(_wesad_path, "r") as _f: _wp = _j2.load(_f)["best_params"]
with open(_sp_path,    "r") as _f: _sp = _j2.load(_f)["best_params"]

# ── Ejecutar para WESAD ────────────────────────────────────────────────────────
print("="*70)
print("EXPERIMENTO 20/80 — CV POR PACIENTE — WESAD")
print("="*70)
results_8020_wesad, plog_wesad = run_split8020_per_subject(
    X, y, groups,
    use_ssl=_wp.get("use_ssl", True),
    ssl_epochs=_wp.get("ssl_epochs", 6),
    ft_epochs=_wp.get("ft_epochs", 8),
    emb_dim=_wp.get("emb_dim", 128),
    depth=_wp.get("depth", 4),
    dropout=_wp.get("dropout", 0.1),
    dataset_name="WESAD", save_dir=RESULTS_DIR,
)
summarize("20/80 CV-por-paciente — WESAD", results_8020_wesad,
          txt_path=f"{RESULTS_DIR}/resultados_8020_WESAD_{WINDOW_SECONDS}seg.txt",
          append=False)

# ── Ejecutar para Stress-ID ────────────────────────────────────────────────────
print("\n" + "="*70)
print("EXPERIMENTO 20/80 — CV POR PACIENTE — STRESS-ID")
print("="*70)
results_8020_sp, plog_sp = run_split8020_per_subject(
    X1, y1, groups_1,
    use_ssl=_sp.get("use_ssl", True),
    ssl_epochs=_sp.get("ssl_epochs", 6),
    ft_epochs=_sp.get("ft_epochs", 8),
    emb_dim=_sp.get("emb_dim", 128),
    depth=_sp.get("depth", 4),
    dropout=_sp.get("dropout", 0.1),
    dataset_name="StressID", save_dir=RESULTS_DIR,
)
summarize("20/80 CV-por-paciente — Stress-ID", results_8020_sp,
          txt_path=f"{RESULTS_DIR}/resultados_8020_StressID_{WINDOW_SECONDS}seg.txt",
          append=False)


EXPERIMENTO 20/80 — CV POR PACIENTE — WESAD
SSL epoch 1/10: 2.7636
SSL epoch 2/10: 2.2950
SSL epoch 3/10: 2.1454
SSL epoch 4/10: 2.0751
SSL epoch 5/10: 1.9789
SSL epoch 6/10: 1.8661
SSL epoch 7/10: 1.8252
SSL epoch 8/10: 1.7739
SSL epoch 9/10: 1.7131
SSL epoch 10/10: 1.6333
FT epoch 1/6: loss=0.3891, val_f1=0.8958
FT epoch 2/6: loss=0.2415, val_f1=0.8515
FT epoch 3/6: loss=0.2190, val_f1=0.8515
FT epoch 4/6: loss=0.2148, val_f1=0.9451
FT epoch 5/6: loss=0.2030, val_f1=0.9451
FT epoch 6/6: loss=0.1921, val_f1=0.8866
  Sujeto   2  |  F1=1.0000  BA=1.0000  AUROC=1.0000  [train_subj=54, test=13]
SSL epoch 1/10: 2.6941
SSL epoch 2/10: 2.2050
SSL epoch 3/10: 2.0541
SSL epoch 4/10: 1.9353
SSL epoch 5/10: 1.8711
SSL epoch 6/10: 1.8241
SSL epoch 7/10: 1.7229
SSL epoch 8/10: 1.6724
SSL epoch 9/10: 1.6418
SSL epoch 10/10: 1.5618
FT epoch 1/6: loss=0.4189, val_f1=0.7073
FT epoch 2/6: loss=0.2316, val_f1=0.7273
FT epoch 3/6: loss=0.1958, val_f1=0.7073
FT epoch 4/6: loss=0.1850, val_f1=0.7368
FT epo

{'accuracy': {'mean': 0.8715969215969216,
  'std': 0.1692976632264189,
  'ci95_low': 0.8115665804875606,
  'ci95_high': 0.9316272627062826,
  'n': 33},
 'f1': {'mean': 0.890355772173954,
  'std': 0.15101445465737628,
  'ci95_low': 0.8368083749284156,
  'ci95_high': 0.9439031694194924,
  'n': 33},
 'balanced_accuracy': {'mean': 0.8684343434343433,
  'std': 0.17488534904744826,
  'ci95_low': 0.8064226950850842,
  'ci95_high': 0.9304459917836024,
  'n': 33},
 'auroc': {'mean': 0.9310606060606061,
  'std': 0.15260004583845638,
  'ci95_low': 0.8769509826292702,
  'ci95_high': 0.9851702294919421,
  'n': 33}}

## Documentación de Sujetos e Índices por Partición 20/80

Tabla completa con los **IDs de sujetos** e **índices de ventanas** asignados a cada partición train/test, para auditoría y reproducibilidad.

In [18]:
# ============================================================
# DOCUMENTACIÓN DE PARTICIONES 20/80 POR SUJETO
# ============================================================
import pandas as pd


def display_partition_log(partition_log, dataset_name):
    """Muestra tabla resumen de sujetos e índices de ventanas por partición.

    Parameters
    ----------
    partition_log : list of dict — devuelto por run_split8020_per_subject.
    dataset_name  : str

    Returns
    -------
    pd.DataFrame con una fila por sujeto.
    """
    rows = []
    for p in partition_log:
        rows.append({
            "Sujeto":              p["subject"],
            "N win train (suj)":   p["n_train_subject"],
            "N win test (suj)":    p["n_test_subject"],
            "N win train (total)": p["n_train_total"],
            "Train cls 0":         p["train_class_dist"].get(0, 0),
            "Train cls 1":         p["train_class_dist"].get(1, 0),
            "Test cls 0":          p["test_class_dist"].get(0, 0),
            "Test cls 1":          p["test_class_dist"].get(1, 0),
            "IDs suj en train":    str(p["train_subjects_used"]),
        })

    df = pd.DataFrame(rows).set_index("Sujeto").sort_index()

    print(f"\n{'='*70}")
    print(f"Particiones 20/80 — {dataset_name}")
    print(f"{'='*70}")
    print(f"  Seed     : {partition_log[0]['seed']}")
    print(f"  test_frac: {partition_log[0]['test_frac']}")
    print(f"  N sujetos: {len(partition_log)}\n")

    try:
        display(df.style.set_caption(
            f"Distribución de ventanas por sujeto — {dataset_name}"
        ))
    except Exception:
        print(df.to_string())

    return df


# ── Tablas para WESAD y Stress-ID ────────────────────────────────────────────
df_parts_wesad = display_partition_log(plog_wesad, "WESAD")
df_parts_sp    = display_partition_log(plog_sp,    "Stress-ID")

# ── Ejemplo detallado de un sujeto ───────────────────────────────────────────
print("\n" + "─"*60)
print("Ejemplo detallado — WESAD")
print("─"*60)
_ej = plog_wesad[0]
print(f"Sujeto {_ej['subject']}:")
print(f"  Sujetos en entrenamiento : {_ej['train_subjects_used']}")
print(f"  Ventanas en TEST  ({len(_ej['test_window_indices'])} total) — primeros 20 índices:")
print(f"    {_ej['test_window_indices'][:20]}")
print(f"  Ventanas en TRAIN del sujeto ({len(_ej['train_window_indices'])} total) — primeros 20 índices:")
print(f"    {_ej['train_window_indices'][:20]}")

print("\n" + "─"*60)
print("Ejemplo detallado — Stress-ID")
print("─"*60)
_ej2 = plog_sp[0]
print(f"Sujeto {_ej2['subject']}:")
print(f"  Sujetos en entrenamiento : {_ej2['train_subjects_used']}")
print(f"  Ventanas en TEST  ({len(_ej2['test_window_indices'])} total) — primeros 20 índices:")
print(f"    {_ej2['test_window_indices'][:20]}")
print(f"  Ventanas en TRAIN del sujeto ({len(_ej2['train_window_indices'])} total) — primeros 20 índices:")
print(f"    {_ej2['train_window_indices'][:20]}")



Particiones 20/80 — WESAD
  Seed     : 42
  test_frac: 0.2
  N sujetos: 15



,N win train (suj),N win test (suj),N win train (total),Train cls 0,Train cls 1,Test cls 0,Test cls 1,IDs suj en train
Sujeto,,,,,,,,
2,54,13,1047,38,16,9,4,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]"
3,55,13,1047,39,16,9,4,"[2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]"
4,56,14,1046,40,16,10,4,"[2, 3, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]"
5,57,13,1047,41,16,10,3,"[2, 3, 4, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]"
6,57,13,1047,40,17,9,4,"[2, 3, 4, 5, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]"
7,57,14,1046,41,16,10,4,"[2, 3, 4, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17]"
8,57,14,1046,40,17,10,4,"[2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 14, 15, 16, 17]"
9,57,13,1047,40,17,9,4,"[2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 16, 17]"
10,59,14,1046,41,18,10,4,"[2, 3, 4, 5, 6, 7, 8, 9, 11, 13, 14, 15, 16, 17]"



Particiones 20/80 — Stress-ID
  Seed     : 42
  test_frac: 0.2
  N sujetos: 33



,N win train (suj),N win test (suj),N win train (total),Train cls 0,Train cls 1,Test cls 0,Test cls 1,IDs suj en train
Sujeto,,,,,,,,
2,41,9,1607,16,25,3,6,"[3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"
3,34,8,1608,14,20,3,5,"[2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"
4,38,8,1608,14,24,3,5,"[2, 3, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"
5,41,10,1606,17,24,4,6,"[2, 3, 4, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"
7,41,9,1607,20,21,4,5,"[2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"
8,38,8,1608,16,22,3,5,"[2, 3, 4, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"
9,36,7,1609,16,20,3,4,"[2, 3, 4, 5, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"
10,35,8,1608,14,21,3,5,"[2, 3, 4, 5, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"
11,41,9,1607,16,25,3,6,"[2, 3, 4, 5, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]"



────────────────────────────────────────────────────────────
Ejemplo detallado — WESAD
────────────────────────────────────────────────────────────
Sujeto 2:
  Sujetos en entrenamiento : [3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
  Ventanas en TEST  (13 total) — primeros 20 índices:
    [569, 530, 535, 534, 528, 566, 508, 567, 521, 544, 558, 552, 559]
  Ventanas en TRAIN del sujeto (54 total) — primeros 20 índices:
    [526, 507, 524, 560, 510, 565, 532, 512, 533, 523, 561, 519, 562, 527, 518, 531, 564, 529, 513, 506]

────────────────────────────────────────────────────────────
Ejemplo detallado — Stress-ID
────────────────────────────────────────────────────────────
Sujeto 2:
  Sujetos en entrenamiento : [3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
  Ventanas en TEST  (9 total) — primeros 20 índices:
    [14, 9, 17, 27, 45, 25, 49, 44, 30]
  Ventanas en TRAIN del sujeto (41 total) — primeros 20 índi